# Memmingen -- Unabhängige Druck-Validierung mit pandapipes

**Ziel**: Unser Pyomo/Gurobi-MILP berechnet Druck/Pumpen-Physik über eine
**linearisierte** Näherung (konvexe Tangenten-/Sekanten-Hüllkurven, siehe
`docs/paper_2/CALION_Paper2_Implementation_Statement.md` Part H). Das ist
nötig, damit die Physik gleichzeitig mit Dispatch/Investitionsentscheidungen
in EINEM MILP lösbar bleibt -- kostet aber etwas an physikalischer Exaktheit.

**pandapipes** ist ein eigenständiger, nicht-linearer hydraulischer Solver
(Newton-Raphson, echte Colebrook-White-Reibung, Reynolds-abhängig) -- er kann
keine Investitions-/Dispatch-Entscheidungen treffen, aber für EINEN gegebenen
Betriebspunkt (feste Flüsse, feste Erzeugerdrücke) liefert er eine praktisch
"exakte" Referenzlösung.

**Methode dieses Notebooks**: Wir lösen unser MILP für eine echte Woche,
extrahieren die GELÖSTEN Flüsse zu einer bestimmten Stunde, füttern diese
EXAKT GLEICHEN Flüsse in ein äquivalentes pandapipes-Netz (gleiche Topologie,
echte Rohrlängen/-durchmesser aus `Memmingen_pressure_stations.yaml`), lösen
pandapipes' echte nicht-lineare Gleichungen, und vergleichen Knoten für
Knoten. Das isoliert GENAU die Frage "wie gut ist unsere Linearisierung",
ohne dass unterschiedliche Randbedingungen das Ergebnis verfälschen.

**Wichtiger Hinweis vorab**: Beim Bauen dieses Notebooks wurde ein echter
Modellierungsfehler gefunden und behoben (Cell 8) -- ein gutes Beispiel dafür,
warum man ein Vergleichsmodell mit Sorgfalt aufbauen muss. Details dort.

Physik-Referenz für alles Druck/Pumpen-bezogene in unserem eigenen Modell:
`docs/paper_2/CALION_Paper2_Implementation_Statement.md` Part H. Reale
Zuleitungs-Verluste (Cell 14+): siehe memory
`project_memmingen_pump_pressure_study` (falls du Zugriff auf die
Session-Memory hast) für die volle Herleitung.

## Zusammenfassung: Wie wir Druck/Pumpen-Physik berechnen -- und wie gut das ist

Diese Zelle ist die Kurzfassung für alle, die nicht jede Code-Zelle lesen
wollen. Details und Belege stehen in den Zellen darunter.

### 1. Die physikalische Grundlage

Der Druckverlust in einem Rohr folgt Darcy-Weisbach:

```
Δp = f · (L / D) · (ρ / 2) · v²         (v = Fließgeschwindigkeit, quadratisch im Durchfluss)
```

`f` (Reibungsfaktor) hängt eigentlich von der Reynolds-Zahl und der relativen
Rohrrauheit ab (Colebrook-White-Gleichung, nicht-linear). Die Pumpenleistung
folgt aus dem Förderhöhen-Wirkungsgrad-Zusammenhang und ist **kubisch** im
Durchfluss (Δp · V̇ / η).

### 2. Warum wir NICHT direkt Darcy-Weisbach lösen

Unser Modell muss Druck/Pumpen-Physik GLEICHZEITIG mit
Investitions-/Dispatch-Entscheidungen (Kapazitäten, An/Aus-Schaltungen,
Speicher-Fahrweise) in EINEM gemischt-ganzzahligen linearen Programm (MILP)
lösen. Eine echte quadratische/kubische Gleichung würde das Problem
nicht-linear (NLP) oder erheblich schwerer machen (Binärvariablen für jede
Kombination). Deshalb linearisieren wir:

- **Konvexe Tangenten-Hüllkurve, ohne Binärvariablen**: An mehreren
  Stützpunkten des Durchflusses (33/67/100% der Rohr-Auslegungskapazität,
  siehe `pipe_pair.py`) werden Tangenten an die echte quadratische
  Δp(ṁ)-Kurve gelegt und als lineare UNTERGRENZEN eingetragen. Weil `Δp`
  (über die Pumpenkosten) im Zielfunktionswert mit einem POSITIVEN
  Koeffizienten steht, hat der Solver einen echten Anreiz, `Δp` so klein wie
  möglich zu halten -- und landet dadurch automatisch auf der straffsten
  bindenden Tangente, ohne dass man ihm mit Binärvariablen vorschreiben
  muss, welches Segment gilt. Dieselbe Idee gilt für `P_pump(ṁ)` (kubisch).
- **Verdichtung bei niedrigem Durchfluss** (`add_low_flow_tangents()`,
  Cell 2): die 3 Standard-Tangenten allein werden unterhalb von ca. 22% der
  Auslegungskapazität nicht-bindend (die Tangente an einem hohen
  Referenzpunkt liegt bei kleinem `ṁ` unter Null) -- typisch für die
  meisten echten Betriebsstunden. Ohne zusätzliche Tangenten bei 2/5/8/12/
  18/25% der Kapazität könnte `Δp`/`P_pump` an der Variablen-Untergrenze (0)
  hängen bleiben, statt den echten (kleinen) Wert zu zeigen.
- **Zuleitungs-Verluste** (`lateral_dp_extra`, kleine DN32-Rohre von der
  Trasse zur Übergabestation): eigene, separate PWL mit gewichteter
  Stützpunkt-Kombination (konvexe Kombination, siehe Pumpstudie-Notebook).
- **Erzeuger-/Fortpflanzungs-Druck**: der Haupterzeuger (`j_9`) hat einen
  FEST vorgegebenen `setpoint_bar`. Stromabwärts wird der Druck über
  Ungleichungen fortgepflanzt (`P_to <= P_from - Δp`, mit etwas Spielraum
  statt strikter Gleichheit, aus numerischen Gründen). Ein optionaler
  `pressure_regularization`-Term (kleiner Zielfunktions-Koeffizient) drängt
  Verbraucherknoten zum straffsten (physikalisch korrekten) Wert, statt sie
  beliebig im Spielraum sitzen zu lassen.

### 3. Wie wir das gegen pandapipes prüfen

Wir lösen unser MILP für eine echte Woche, extrahieren die GELÖSTEN Flüsse
zu einer festen Stunde, und füttern EXAKT DIESE Flüsse in ein äquivalentes
pandapipes-Netz (gleiche Topologie, echte Rohrlängen/-durchmesser).
pandapipes löst dann die ECHTEN nicht-linearen Gleichungen (Newton-Raphson,
Colebrook-White) für genau diesen einen Betriebspunkt. Das isoliert exakt
die Frage "wie gut ist unsere Linearisierung" -- pandapipes trifft dabei
selbst keine Dispatch-Entscheidung, sondern ist reiner Referenz-Löser.

### 4. Ergebnis in einem Satz

Für die große Mehrheit des Netzes (Trassenrohre, 9 von 15 Knoten) stimmt
unsere Linearisierung mit der echten nicht-linearen Lösung auf
**< 0.01 bar** überein -- sehr gut. Für kleine Zuleitungsrohre und für einen
sekundären Erzeugerknoten (`j_12`) gibt es bekannte, unten quantifizierte
Lücken. Details und Zahlen: siehe Cells 9-15 und das **Fazit** am Ende --
dort steht auch, WELCHE Zahlen aus diesem und dem Vollständigkeitshalber
später ergänzten Ganzjahres-Lauf man trauen kann und welche nicht.

## Detaillierte Methodik: Wie jede Druck-/Pumpengröße berechnet wird

Diese Sektion geht deutlich tiefer als die Kurzfassung oben -- mit den
tatsächlichen Formeln, den exakten Constraint-Definitionen aus dem Code,
und den Grenzen jeder Vereinfachung. Referenzierter Code:
`calion/models/blocks/pipe_pair.py`, `calion/models/blocks/thermal_node.py`,
`calion/models/network_manager.py`.

### 1. Physikalische Grundgleichung (Darcy-Weisbach)

Reibungsdruckverlust in einem Rohr:

```
Δp = f · (L/D) · (ρ/2) · v²
```

Mit `v = ṇ/(ρ·A)` (Massenstrom durch Querschnittsfläche) wird das zu einer
reinen Funktion des Massenstroms:

```
Δp(ṇ) = k_flow · ṇ²,      k_flow = f · L / (2 · ρ · D · A²)
```

`f` (Reibungsfaktor) hängt über die Colebrook-White-Gleichung eigentlich
von der Reynolds-Zahl und der relativen Rohrrauheit ab (nicht-linear) --
wir nehmen ihn KONSTANT an (Standard `f=0.02`). Teil 1 validiert diese
Annahme unabhängig gegen pandapipes: sehr gut für Trassenrohre (echter Wert
0.0166-0.0221), ~30-45% zu niedrig für die kleinen DN32-Zuleitungen (echter
Wert 0.027-0.030).

Pumpleistung (mechanisch, mit Wirkungsgrad η):

```
P_pump(ṇ) = Δp · V̇ / η = Δp · ṇ / (ρ·η) = k_flow · ṇ³ / (ρ·η) = c_pump · ṇ³
```

-- **kubisch** im Massenstrom, nicht quadratisch wie Δp selbst.

### 2. Trassen-Linearisierung: konvexe Tangenten-Hüllkurve

`Δp(ṇ)` und `P_pump(ṇ)` sind konvexe Funktionen, UND beide stehen im
Zielfunktionswert mit einem POSITIVEN Kostenkoeffizienten (die Pumpenkosten
sollen minimiert werden). Für eine solche Konstellation reicht eine Menge
linearer UNTERGRENZEN (Tangenten an mehreren Stützpunkten `ṇ_i`), um die
exakte konvexe Funktion am Optimum zu reproduzieren -- **ohne
Binärvariablen oder SOS2**:

```
Tangente an ṇ_i:   Δp(ṇ)     ≥ 2·k_flow·ṇ_i·ṇ  -   k_flow·ṇ_i²
                    P_pump(ṇ) ≥ 3·c_pump·ṇ_i²·ṇ - 2·c_pump·ṇ_i³
```

(jeweils die Tangente der Parabel/Kubik am Punkt `ṇ_i` -- Standard-Technik
für konvexe Piecewise-Linearisierung ohne Ganzzahligkeit). `pipe_pair.py`
setzt 3 Standard-Stützpunkte bei 33/67/100% der Rohr-Auslegungskapazität.

**Warum das (meistens) exakt ist**: der Solver hat einen echten Anreiz,
`Δp`/`P_pump` so klein wie möglich zu halten (Minimierungsrichtung + echte
Kosten) -- er landet deshalb automatisch auf der straffsten bindenden
Tangente, was GENAU dem wahren Funktionswert entspricht, keine Näherung.
**Wo das bricht**: bei sehr niedrigem Durchfluss (unter ca. 22% der
Auslegungskapazität) liegen alle 3 Standard-Tangenten unterhalb von Null
und werden nicht-bindend -- `add_low_flow_tangents()` (in diesem Notebook
verwendet) fügt deshalb 6 weitere Stützpunkte bei 2/5/8/12/18/25% hinzu.
**Trotzdem nicht vollständig geschlossen**: Teil "Nachtrag" unten zeigt,
dass die Pumpenleistung übers Jahr nur schwach (r=0.41) mit dem echten
Fluss korreliert -- diese Verdichtung reicht nicht in jedem Fall.

**Korrektur (2026-08-03, gefunden beim Bauen von Teil 6)**: Die obige
Beschreibung (Tangenten-Untergrenzen für `P_pump`) ist die **Opt-out**-Variante
(`pump_pin_pwl=false`, ein Escape-Hatch). Der tatsächliche STANDARD in
`pipe_pair.py` (`pump_pin_pwl=True`) ist genauer: `P_pump` wird über eine
EXAKTE, binär-segmentierte PWL-GLEICHUNG gepinnt (5 Stützpunkte, echte
binäre Segment-Auswahl mit engen Flow-Schranken je Segment) -- nicht über
lockere Tangenten-Untergrenzen. Das ist robuster (kann bei Teillast weder
auf 0 kollabieren noch bei negativen Strompreisen künstlich hochschießen)
und kann NICHT das "nicht-benachbarte Stützpunkte"-Gewichtungs-Artefakt
zeigen, das bei `lateral_dp_extra` (Abschnitt 3) gefunden wurde, weil echte
Binärvariablen (kein freier Gewichtungsvektor) das Segment eindeutig
festlegen. **Offene Frage, NICHT in dieser Sitzung geklärt**: dieser
präzisere Mechanismus scheint der pipe_pair.py-Standard zu sein -- ob die
in Teil 1-5 zusätzlich per `add_low_flow_tangents()` hinzugefügten
`P_pump`-Tangenten dann noch einen Effekt haben (vermutlich harmlos-redundant,
da eine echte Tangente einer konvexen Funktion nie über deren
Sekanten-Interpolation liegen kann) oder ob die ursprünglich gefundene
"Pumpenleistung korreliert nur schwach mit dem Fluss" (Nachtrag, Punkt 7)
unter dem STANDARD-Mechanismus überhaupt noch auftritt, wurde NICHT erneut
geprüft. Teil 6 verwendet deshalb bewusst NUR die `delta_p`-Tangenten
(kein `P_pump`-Zusatzterm) -- konsistent mit dem Original-Notebook
(`Memmingen_pump_pressure_study.ipynb`, Cell 6b).

**Audit (2026-08-03), nach der `pump_pin_pwl`-Korrektur oben: systematisch
nach weiteren versteckten Default-Abweichungen dieser Art gesucht.**
Direkt im Code geprüft: `delta_p_supply` (Trasse -- bestätigt: nur 3
Tangenten bei 33/67/100%, KEIN "pin"-Äquivalent, siehe unten für den echten
Fehlerwert), `lateral_dp_extra` (bestätigt: weiterhin gewichtete
11-Stützpunkt-Kombination, kein Binär-Upgrade), `P_return`-Regularisierung
(bestätigt: laut Code-Kommentar in `network_manager.py` ABSICHTLICH nicht
umgesetzt, "needs its own correctly-anchored formulation later" -- kein
verstecktes Feature übersehen), TES-Druckkopplung (`same_circuit_buffer`:
bestätigt nur eine einfache Behälter-Druckobergrenze, keine versteckte
Kopplung an `P_supply`/`P_return`), bidirektionale Rohre (keine in
Memmingen vorhanden) und `pump_enabled` (kein Override, überall aktiv).
**Ergebnis: keine weitere Abweichung dieser Größenordnung gefunden** -- der
`pump_pin_pwl`-Fund bleibt der einzige.

**Eine kleine, echte Ergänzung dabei gefunden** (keine Korrektur, nur mehr
Präzision): die 3 Trassen-Tangenten wurden bewusst von ursprünglich 5 auf 3
reduziert (Kommentar in `pipe_pair.py`: "halves the pressure-envelope row
count ... to speed the barrier root"), mit einem dokumentierten Worst-Case-
Fehler von **~7% Unterschätzung zwischen den Tangentenpunkten** (~0.03-0.14
bar auf Druckabfällen unter 2 bar) -- klein gegen den 0.6 bar
Stations-Differenzdruck, aber ein reales, bewusstes Genauigkeits-für-
Geschwindigkeit-Trade-off, das in Teil 1/2s pandapipes-Validierung
(< 0.01 bar RMSE) bereits mit abgedeckt ist, hier nur explizit benannt.

### 3. Zuleitungsverlust: gewichtete Stützpunkt-Kombination

Für die kleinen DN32-Zuleitungsrohre (Trasse → Übergabestation) wird
dieselbe konvexe `Δp(ṇ)`-Funktion NICHT über Tangenten, sondern über eine
explizite konvexe Kombination von (Stützpunkt, Funktionswert)-Paaren
dargestellt (`thermal_node.py`):

```
lateral_dp_extra = Σ_k λ_k · Δp(ṇ_k),   mit   ṇ = Σ_k λ_k · ṇ_k,
                    Σ_k λ_k = 1,   λ_k ≥ 0
```

`ṇ` hier ist der Durchfluss PRO STATION (`m_dot_demand / n_transfer_stations`),
nicht der Knoten-Gesamtfluss -- jede reale Zuleitung trägt nur den Anteil
EINER Übergabestation. Für eine konvexe Funktion, die nur in
Minimierungsrichtung gebraucht wird, ist auch das exakt darstellbar ohne
Binärvariablen -- SOLANGE der Kostenanreiz stark genug ist. **Genau hier
liegt der im "Nachtrag" dokumentierte Solver-Artefakt**: bei einem
lockeren MIP-Gap kann sich die Gewichtung `λ_k` auf NICHT-benachbarte
Stützpunkte verteilen -- eine gültige konvexe Kombination (die
Fluss-Kopplung bleibt erfüllt), aber mit einem stark überhöhten
interpolierten Wert (bis 23.6-fach beobachtet). Der Tie-Break-Fix
(`lateral_tiebreak_epsilon`) mildert, schließt aber nicht vollständig.

### 4. Druckfortpflanzung im Netz (`network_manager.py::_link_pressure_propagation`)

- **Haupterzeuger** (`j_9`): `P_supply` FEST auf den Sollwert (6.38 bar --
  reales Δp-c-Kurvenmaximum der installierten Wilo IL-E 65/11-64 BF Pumpe).
- **Sekundärer Erzeuger** (`j_12`): `P_supply` FREI mit einem
  `head_max`-Fußpunkt. **Update (2026-08-03)**: zusätzlich jetzt per
  topologie-skaliertem Tie-Break auf seinen eigenen, echten Fußpunkt
  gezogen (siehe Fazit Kategorie B.5 für die vollständige Herleitung) --
  die frühere `j_12`-Ceiling-Lücke ist damit behoben, nicht mehr nur
  dokumentiert.
- **Alle anderen (propagierten) Knoten**: Ungleichung statt Gleichheit
  (numerische Gründe, PWL-Kompatibilität):
  ```
  P_supply[Knoten]   ≤ P_supply[Vorgänger] - Δp_Trasse(ṇ)
  P_return[Vorgänger] ≤ P_return[Knoten]   - Δp_Trasse_Rücklauf(ṇ)
  ```
- **Regularisierung** (`pressure_regularization`, opt-in, hier AN,
  `epsilon=1e-4`): ein kleiner Zielfunktions-Term (`-epsilon · P_supply[Knoten]`)
  drängt jeden propagierten Knoten dazu, den Spielraum der obigen
  Ungleichung voll auszunutzen (`P_supply` so HOCH wie erlaubt), statt
  beliebig darunter zu liegen -- ohne das wäre `P_supply` an jedem nicht
  direkt bindenden Knoten ein Solver-Artefakt (bekanntes Beispiel: `j_12`
  und alles stromabwärts, wo dieser Mechanismus bewusst NICHT greift).
- **`P_return` wird NICHT regularisiert** -- ein früherer Versuch drängte
  es ohne Anker an den Netz-Enden auf eine willkürliche Obergrenze. Das ist
  wichtig für Teil 5 unten: die dortige Prüfung hat dadurch für die
  meisten Stunden wenig Aussagekraft.

### 5. Minimaldruck an Verbraucherknoten -- ZWEI SEPARATE, UNVERKNÜPFTE Constraints

**(a) Absoluter Fußpunkt** (`network_manager.py`):
```
P_supply[Knoten] ≥ min_required_bar     (hier: 2.0 bar, für alle Verbraucher gleich)
```
Grundlage für die Margin-Analyse in Teil 3.

**(b) Differenzdruck an der Übergabestation** (`thermal_node.py::_station_dp_rule`):
```
P_supply[Knoten] - P_return[Knoten] ≥ delta_p_min_consumer_bar + lateral_dp_extra[Knoten]
```
mit `delta_p_min_consumer_bar = 0.6 bar` (60 kPa) -- **ein echter
Spezifikationswert**, nicht angenommen: `Netzkomponenten_Spezifikationen.xlsx`,
Blatt "Uebergabestation", listet sowohl "min Differenzdruck Waermenetz" als
auch "Druckverluste" mit 0.6 bar. Das ist die eigentliche physikalische
Anforderung des Regelventils an der Station. Teil 5 unten prüft das direkt.

**Wichtig**: `min_required_bar` (2.0 bar) ist NICHT als "0.6 + typischer
Zuleitungsverlust" hergeleitet -- es ist ein unabhängig gewählter,
konstanter Fußpunkt, gleich für alle Knoten trotz sehr unterschiedlicher
Zuleitungsverluste (0.02 bis 0.6+ bar je Knoten, siehe Teil 3). Die beiden
Constraints (a) und (b) können also unterschiedlich knapp sein -- (a) ist
in Teil 3/4 gezeigt, (b) erst in Teil 5 unten.

### 6. Pumpenleistungs-Kosten (`network_manager.py::_link_pump_head`)

Jeder Erzeuger (`j_9`, `j_12`) übernimmt die Pumpenleistung für jedes Rohr
auf seinem eigenen Ast (Multi-Source-BFS vom Erzeuger, endet an der
nächsten Erzeuger-Grenze) PLUS die Stations-/Zuleitungspumparbeit:
```
producer_P_pump_total = Σ_Rohre P_pump(Rohr)
                       + Σ_Knoten [ delta_p_min_consumer_bar · m_dot_demand
                                     + P_pump_lateral(Knoten) ]
```
Diese Summe fließt als ECHTE Stromlast in den Strombus (`el_in`) --
keine reine Kosten-Buchhaltung, sondern Teil der tatsächlichen
Dispatch-Physik im MILP.

### Cell 1 — Setup: Imports (inkl. pandapipes)

In [ ]:
# =============================================================================
# Cell 1 — Setup
# =============================================================================
import sys
import time
import math as _math
from pathlib import Path

_ROOT = Path(r"c:\Users\LKR\Documents\GitHub\Energy_Framwork\Planing-Framework-for-Heat")
sys.path.insert(0, str(_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyomo.environ as pyo
import pandapipes as pp

from scripts.paper_2.scenario_runner import (
    _load_yaml, _dump_yaml_tmp, _apply_spatial_temperature_offsets,
    _inject_cop_series, _load_outdoor_temps, _load_hp_source_temps,
)
from calion.run.workflow import _build_workflow_inputs
from calion.utils.heizkurve import compute_heizkurve
from calion.utils.cop_wrapper import precompute_cop
from calion.models.system_builder import build_model

CONFIG_PATH = _ROOT / "configs" / "pressure" / "Memmingen_pressure_stations.yaml"
print("Setup OK. pandapipes version:", pp.__version__)


### Cell 2 — Unser MILP lösen (echte Winterwoche)

Gleiche Pipeline wie im Pumpstudie-Notebook. `add_low_flow_tangents()` ist
dieselbe notebook-lokale Verdichtung wie dort -- ohne sie bleiben bei echten
(deutlich unter Auslegungskapazität liegenden) Flüssen alle 3
Original-Tangenten von `pipe_pair.py` nicht-bindend, und `delta_p_supply`
könnte an seiner Variablen-Untergrenze (0) hängen bleiben, statt den echten
(kleinen, aber realen) Reibungsverlust zu zeigen -- das würde diesen
Vergleich verfälschen, bevor er überhaupt anfängt.

In [ ]:
# =============================================================================
# Cell 2 — add_low_flow_tangents() + build_and_solve() helpers
# =============================================================================
def add_low_flow_tangents(model, network_manager, extra_fracs=(0.02, 0.05, 0.08, 0.12, 0.18, 0.25)):
    density_water = 1000.0
    f_friction = 0.02
    max_velocity = float(network_manager._net_cfg.get('max_velocity_m_s', 2.5))
    eta_pump = float(network_manager._net_cfg.get('pump_efficiency', 0.70))
    n_added = 0
    for pipe_id, pipe_cfg in network_manager.pipes.items():
        prefix = pipe_id.upper().replace('-', '_')
        m_dot_var = getattr(model, f'{prefix}_m_dot', None)
        delta_p_supply = getattr(model, f'{prefix}_delta_p_supply', None)
        P_pump_var = getattr(model, f'{prefix}_P_pump', None)
        if m_dot_var is None or delta_p_supply is None:
            continue
        length_m = float(pipe_cfg.get('length_m', 0) or 0)
        diameter_mm = float(pipe_cfg.get('diameter_mm', 0) or 0)
        if length_m <= 0 or diameter_mm <= 0:
            continue
        d_inner_m = diameter_mm / 1000.0 * 0.94
        area_m2 = _math.pi * (d_inner_m / 2.0) ** 2
        effective_max_flow = area_m2 * max_velocity * density_water
        k_pressure = f_friction * (length_m / d_inner_m) * (density_water / 2.0) / 1e5
        k_flow = k_pressure / ((density_water * area_m2) ** 2)
        c_pump = 2.0 * k_flow * 1e5 / (density_water * eta_pump * 1e6)
        for frac in extra_fracs:
            mi = frac * effective_max_flow
            if mi <= 0:
                continue
            setattr(model, f'{prefix}_dp_lowflow_{int(frac*1000)}',
                    pyo.Constraint(model.t, rule=(
                        lambda m, t, _mdv=m_dot_var, _dp=delta_p_supply, _mi=mi, _kf=k_flow:
                        _dp[t] >= 2.0 * _kf * _mi * _mdv[t] - _kf * _mi ** 2
                    )))
            n_added += 1
            if P_pump_var is not None:
                setattr(model, f'{prefix}_pp_lowflow_{int(frac*1000)}',
                        pyo.Constraint(model.t, rule=(
                            lambda m, t, _mdv=m_dot_var, _pp=P_pump_var, _mi=mi, _c=c_pump:
                            _pp[t] >= 3.0 * _c * _mi ** 2 * _mdv[t] - 2.0 * _c * _mi ** 3
                        )))
                n_added += 1
    return n_added


def build_and_solve(horizon_start, horizon_end, label, time_limit_s=90, mip_gap=0.03):
    cfg = _load_yaml(CONFIG_PATH)
    cfg["scenario"]["horizon"] = {"start": horizon_start, "end": horizon_end}
    tmp1 = _dump_yaml_tmp(cfg)
    inputs0 = _build_workflow_inputs([str(tmp1)], overrides=None)
    table = inputs0.table
    try:
        tmp1.unlink()
    except OSError:
        pass
    hc = cfg["network"]["heating_curve"]
    T_aus = _load_outdoor_temps(table, cfg)
    T_VL_ts = compute_heizkurve(k=hc["k"], T_VL_min_c=hc["T_supply_min_c"], T_VL_max_c=hc["T_supply_max_c"], T_aus_ts=T_aus)
    return_temp_c = float(cfg.get("network", {}).get("return_temp_c", 60.0))
    min_delta_T = float(cfg.get("network", {}).get("min_supply_delta_T_k", 10.0))
    T_VL_min_effective = max(float(hc["T_supply_min_c"]), return_temp_c + min_delta_T)
    T_VL_ts = np.maximum(T_VL_ts, T_VL_min_effective)
    cfg["network"]["heating_curve"]["T_supply_min_c"] = T_VL_min_effective
    cfg.setdefault("heat_pumps", {}).setdefault("cop", {})
    cfg["heat_pumps"]["cop"]["supply_temp_min_c"] = T_VL_min_effective
    cfg["heat_pumps"]["cop"]["supply_temp_max_c"] = hc["T_supply_max_c"]
    delta_T_scenario_k = round(T_VL_min_effective - return_temp_c, 2)
    for _ak, _acfg in cfg.get("assets", {}).items():
        if _acfg.get("type") == "geometric_storage":
            _acfg["delta_T_scenario_k"] = delta_T_scenario_k
    T_source_ts = _load_hp_source_temps(cfg, table)
    cop_ts = precompute_cop(T_VL_ts=T_VL_ts, T_source_ts=T_source_ts, table=table, cfg=cfg, hp_type="standard")
    _inject_cop_series(cfg, cop_ts)
    _apply_spatial_temperature_offsets(cfg, T_VL_ts, f"MM-PP-{label}")
    tmp = _dump_yaml_tmp(cfg)
    inputs = _build_workflow_inputs([str(tmp)], overrides=None)
    try:
        tmp.unlink()
    except OSError:
        pass
    model = build_model(inputs.table, inputs.cfg, dt_h=inputs.dt_h)
    n_tangents = add_low_flow_tangents(model, model._network_manager)
    print(f"[{label}] build OK, added {n_tangents} low-flow tangent constraints")
    solver_options = dict(inputs.cfg.get("run", {}).get("solver_options", {}))
    solver_options["TimeLimit"] = time_limit_s
    solver_options["MIPGap"] = mip_gap
    opt = pyo.SolverFactory(inputs.solver_name)
    for k, v in solver_options.items():
        opt.options[k] = v
    res = opt.solve(model, tee=False, warmstart=False, load_solutions=False)
    n_sol = len(res.solution) if hasattr(res, "solution") else 0
    print(f"[{label}] termination={res.solver.termination_condition}, n_sol={n_sol}")
    if n_sol == 0:
        raise RuntimeError(f"[{label}] no incumbent")
    model.solutions.load_from(res)
    return model, cfg, inputs


model, cfg, inputs = build_and_solve("2025-12-29 00:00", "2026-01-04 23:00", "WINTER", time_limit_s=90)


### Cell 3 — Zwei Betriebspunkte wählen: Spitzenlast + mittlere Last

Damit der Vergleich nicht nur an einem einzigen (evtl. untypischen) Punkt
gilt, prüfen wir sowohl die höchste als auch eine mittlere Gesamtlast-Stunde
innerhalb derselben Woche.

In [ ]:
# =============================================================================
# Cell 3 — Pick peak-demand and median-demand hours
# =============================================================================
ts = list(model.t)
PRIMARY = cfg["network"]["primary_producer"]

total_flow_per_t = []
for t in ts:
    total = 0.0
    for nid in cfg["network"]["nodes"]:
        prefix = nid.upper().replace('-', '_')
        m_dot = getattr(model, f"{prefix}_m_dot_demand", None)
        if m_dot is not None:
            total += pyo.value(m_dot[t])
    total_flow_per_t.append(total)

peak_idx = int(np.argmax(total_flow_per_t))
peak_t = ts[peak_idx]
median_idx = int(np.argsort(total_flow_per_t)[len(total_flow_per_t) // 2])
median_t = ts[median_idx]

print(f"Peak-demand hour:   index={peak_idx}, t={peak_t}, total_flow={total_flow_per_t[peak_idx]:.2f} kg/s")
print(f"Median-demand hour: index={median_idx}, t={median_t}, total_flow={total_flow_per_t[median_idx]:.2f} kg/s")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(total_flow_per_t)
ax.axvline(peak_idx, color="red", ls="--", label="peak")
ax.axvline(median_idx, color="orange", ls="--", label="median")
ax.set_ylabel("Gesamtdurchfluss [kg/s]")
ax.set_title("Gesamt-Netzdurchfluss über die Winterwoche")
ax.legend()
plt.show()


### Cell 4 — Gelöste Daten extrahieren

Für jeden Knoten: `pressure_supply`, `pressure_return`, `m_dot_demand`
(Verbrauch), `m_dot_gen` (**lokale Erzeugung** -- siehe Cell 8, warum das
entscheidend ist), `lateral_dp_extra` (falls die reale Zuleitungs-Verlust-
Funktion für diesen Knoten aktiv ist). Für jedes Rohr: `m_dot`,
`delta_p_supply`.

In [ ]:
# =============================================================================
# Cell 4 — extract_hour(): pull everything we need from the solved model
# =============================================================================
def extract_hour(model, cfg, t):
    data = {"nodes": {}, "pipes": {}}
    for nid in cfg["network"]["nodes"]:
        prefix = nid.upper().replace('-', '_')
        p_sup = getattr(model, f"{prefix}_pressure_supply", None)
        p_ret = getattr(model, f"{prefix}_pressure_return", None)
        m_dot_demand = getattr(model, f"{prefix}_m_dot_demand", None)
        lat_dp = getattr(model, f"{prefix}_lateral_dp_extra", None)
        # Local generation (heat pump / e-boiler at a secondary 'mixed'
        # producer like j_12) injects ADDITIONAL mass flow right at that
        # node -- constraint_builder.py's shared m_dot_gen Var/Expression.
        m_dot_gen = getattr(model, f"{prefix}_m_dot_gen", None)
        data["nodes"][nid] = {
            "p_supply_bar": pyo.value(p_sup[t]) if p_sup is not None else None,
            "p_return_bar": pyo.value(p_ret[t]) if p_ret is not None else None,
            "m_dot_demand_kg_s": pyo.value(m_dot_demand[t]) if m_dot_demand is not None else 0.0,
            "lateral_dp_extra_bar": pyo.value(lat_dp[t]) if lat_dp is not None else 0.0,
            "m_dot_gen_kg_s": pyo.value(m_dot_gen[t]) if m_dot_gen is not None else 0.0,
        }
    for pid, pcfg in cfg["network"]["pipes"].items():
        pprefix = pid.upper().replace('-', '_')
        m_dot = getattr(model, f"{pprefix}_m_dot", None)
        dp_sup = getattr(model, f"{pprefix}_delta_p_supply", None)
        data["pipes"][pid] = {
            "from": pcfg["from"], "to": pcfg["to"],
            "length_m": pcfg["length_m"], "diameter_mm": pcfg["diameter_mm"],
            "m_dot_kg_s": pyo.value(m_dot[t]) if m_dot is not None else None,
            "delta_p_supply_bar": pyo.value(dp_sup[t]) if dp_sup is not None else None,
        }
    return data


peak_data = extract_hour(model, cfg, peak_t)
median_data = extract_hour(model, cfg, median_t)
print("Extracted peak-hour and median-hour data for", len(peak_data["nodes"]), "nodes,",
      len(peak_data["pipes"]), "pipes.")


### Cell 5 — Prüfung 1: Massenbilanz VOR dem Bau des pandapipes-Netzes

Bevor wir überhaupt pandapipes anfassen: stimmt die extrahierte Bilanz
`Σ Verbrauch = Σ lokale Erzeugung + Netto-Bezug vom Erzeuger` in UNSEREM
eigenen Modell? Das ist keine pandapipes-Prüfung, sondern eine Kontrolle,
dass wir die richtigen Zahlen extrahiert haben, bevor wir sie weiterreichen.

In [ ]:
# =============================================================================
# Cell 5 — Sanity check: our OWN extracted mass balance closes
# =============================================================================
for label, data in [("PEAK", peak_data), ("MEDIAN", median_data)]:
    total_demand = sum(d["m_dot_demand_kg_s"] for d in data["nodes"].values())
    total_gen = sum(d["m_dot_gen_kg_s"] for d in data["nodes"].values())
    net_from_plant = total_demand - total_gen
    print(f"[{label}] total demand={total_demand:.3f} kg/s, total local generation={total_gen:.3f} kg/s, "
          f"net from plant (expected)={net_from_plant:.3f} kg/s")
    assert total_demand >= 0 and total_gen >= 0, "negative flow -- extraction bug"
print("OK -- both hours have a physically sane (non-negative) demand/generation split.")


### Cell 6 — pandapipes-Netz aufbauen: Topologie

Direkte Entsprechung:
- **Junction** = unser Knoten (gleiche 15 Knoten)
- **Pipe** (`create_pipe_from_parameters`) = unser Rohr, mit den ECHTEN
  `length_m`/`diameter_mm` aus der YAML (gleicher 0.94-Wandstärkefaktor wie
  in unserem eigenen Modell, damit der Innendurchmesser identisch ist)
- **ext_grid** = unser `primary_producer` (`j_9`), fixiert auf den gleichen
  `setpoint_bar` (4.8 bar)
- **sink** = `m_dot_demand` an jedem Verbraucherknoten (EXAKT der von
  unserem MILP gelöste Wert -- nicht neu berechnet)

`k_mm=0.1` (Rohrrauheit) ist eine plausible Annahme für die AGFW-gelabelten
KMR-Rohre in der DXF (moderat glatt, neuere Kunststoffmantelrohr-Bauweise) --
nicht aus der DXF direkt ausgelesen, da diese Info dort nicht vorliegt.

In [ ]:
# =============================================================================
# Cell 6 — build_pandapipes_net(): topology only (generation source comes
# in Cell 8, after we show why it's needed)
# =============================================================================
def build_pandapipes_net(cfg, hour_data, t_supply_k=353.15, include_generation_sources=True):
    net = pp.create_empty_network(fluid="water")
    junction_of = {}
    for nid in cfg["network"]["nodes"]:
        junction_of[nid] = pp.create_junction(net, pn_bar=25, tfluid_k=t_supply_k, name=nid)

    primary = cfg["network"]["primary_producer"]
    p_setpoint = cfg["network"]["nodes"][primary]["pressure"]["setpoint_bar"]
    pp.create_ext_grid(net, junction=junction_of[primary], p_bar=p_setpoint, t_k=t_supply_k, name="plant")

    for pid, pcfg in cfg["network"]["pipes"].items():
        pp.create_pipe_from_parameters(
            net, from_junction=junction_of[pcfg["from"]], to_junction=junction_of[pcfg["to"]],
            length_km=pcfg["length_m"] / 1000.0,
            inner_diameter_mm=pcfg["diameter_mm"] * 0.94,
            k_mm=0.1, name=pid,
        )

    n_sinks = 0
    for nid, ndata in hour_data["nodes"].items():
        m = ndata["m_dot_demand_kg_s"]
        if m and m > 1e-9:
            pp.create_sink(net, junction=junction_of[nid], mdot_kg_per_s=m, name=f"sink_{nid}")
            n_sinks += 1

    n_sources = 0
    if include_generation_sources:
        for nid, ndata in hour_data["nodes"].items():
            m_gen = ndata.get("m_dot_gen_kg_s", 0.0)
            if m_gen and m_gen > 1e-9:
                pp.create_source(net, junction=junction_of[nid], mdot_kg_per_s=m_gen, name=f"source_{nid}")
                n_sources += 1

    print(f"Built pandapipes net: {len(junction_of)} junctions, {len(cfg['network']['pipes'])} pipes, "
          f"{n_sinks} sinks, {n_sources} local-generation source(s), ext_grid at {primary} = {p_setpoint} bar")
    return net, junction_of


### Cell 7 — Erster Versuch: OHNE lokale Erzeugung als Quelle

`j_12` hat eine eigene Wärmepumpe + E-Kessel (`hp_main`, `eboiler_main`) --
das bedeutet, ein TEIL des Massenstroms, der stromabwärts von `j_12`
gebraucht wird, wird NICHT über die Zuleitung `j11_to_j12` herangeführt,
sondern LOKAL bei `j_12` selbst ins Netz eingespeist (dieselbe Physik wie
`constraint_builder.py`s `m_dot_gen`-Korrektur -- siehe dortige Kommentare).
Lassen wir das zunächst absichtlich weg, um zu zeigen, was passiert, wenn man
es vergisst (genau dieser Fehler ist beim Bauen dieses Notebooks passiert).

In [ ]:
# =============================================================================
# Cell 7 — Deliberately WRONG first attempt: no generation source at j_12
# =============================================================================
net_wrong, junction_of = build_pandapipes_net(cfg, peak_data, include_generation_sources=False)
pp.pipeflow(net_wrong, mode="hydraulics")
print("Converged:", net_wrong["converged"])

j12_idx = junction_of["j_12"]
print(f"\nj_12 pressure (WITHOUT generation source): {net_wrong.res_junction.loc[j12_idx, 'p_bar']:.3f} bar")

pipe_idx = {name: idx for idx, name in zip(net_wrong.pipe.index, net_wrong.pipe["name"])}
v_wrong = net_wrong.res_pipe.loc[pipe_idx["j11_to_j12"], "v_mean_m_per_s"]
print(f"j11_to_j12 velocity (WITHOUT generation source): {v_wrong:.3f} m/s "
      f"-- suspiciously high for a {cfg['network']['pipes']['j11_to_j12']['diameter_mm']}mm pipe "
      f"carrying only this node's own small share of demand.")
print("\n==> Diagnosis: without a source at j_12, pandapipes has no way to know that j_12 injects its\n"
      "    own mass flow locally -- it force-routes ALL downstream demand through the trunk pipe\n"
      "    instead, wildly overloading it. This is a MODELING bug in the comparison setup, not a\n"
      "    finding about our MILP. Fixed in Cell 8.")


### Cell 8 — Korrektur: lokale Erzeugung als `source` hinzufügen

`pandapipes.create_source()` speist Massenstrom an einer Junction ein --
genau das Gegenteil von `create_sink()`. Mit `m_dot_gen` aus unserem eigenen
Modell (Cell 4) als Quelle bei `j_12` sollte sich das Bild deutlich normalisieren.

In [ ]:
# =============================================================================
# Cell 8 — Corrected: WITH generation source
# =============================================================================
net_peak, junction_of = build_pandapipes_net(cfg, peak_data, include_generation_sources=True)
pp.pipeflow(net_peak, mode="hydraulics")
print("Converged:", net_peak["converged"])

pipe_idx = {name: idx for idx, name in zip(net_peak.pipe.index, net_peak.pipe["name"])}
v_fixed = net_peak.res_pipe.loc[pipe_idx["j11_to_j12"], "v_mean_m_per_s"]
print(f"j11_to_j12 velocity (WITH generation source): {v_fixed:.3f} m/s (was {v_wrong:.3f} m/s -- much more plausible now)")

# Mass-balance check: ext_grid should supply exactly (total demand - total local generation)
total_sink = sum(d["m_dot_demand_kg_s"] for d in peak_data["nodes"].values())
total_source = sum(d.get("m_dot_gen_kg_s", 0.0) for d in peak_data["nodes"].values())
expected_ext = -(total_sink - total_source)
actual_ext = net_peak.res_ext_grid["mdot_kg_per_s"].values[0]
print(f"\nMass balance check: ext_grid={actual_ext:.4f} kg/s, expected={expected_ext:.4f} kg/s "
      f"(diff={abs(actual_ext - expected_ext):.6f} kg/s)")
assert abs(actual_ext - expected_ext) < 1e-3, "mass balance does not close -- something is still wrong"
print("PASS -- mass balance closes to within 1e-3 kg/s.")


### Cell 9 — Vergleich: Knotendrücke, Spitzenlaststunde

Das ist der Kernvergleich: unser MILP's `pressure_supply` gegen pandapipes'
vollständig nicht-lineare Lösung, Knoten für Knoten, bei EXAKT denselben
Flüssen.

In [ ]:
# =============================================================================
# Cell 9 — Node pressure comparison, peak hour
# =============================================================================
p_setpoint = cfg["network"]["nodes"][cfg["network"]["primary_producer"]]["pressure"]["setpoint_bar"]
rows = []
for nid, jidx in junction_of.items():
    our_p = peak_data["nodes"][nid]["p_supply_bar"]
    pp_p = net_peak.res_junction.loc[jidx, "p_bar"]
    diff = pp_p - our_p
    rows.append({"node": nid, "our_MILP_bar": our_p, "pandapipes_bar": pp_p,
                 "diff_bar": diff, "diff_pct_of_setpoint": diff / p_setpoint * 100})
df_compare_peak = pd.DataFrame(rows).sort_values("node")
print(df_compare_peak.to_string(index=False))
print(f"\nMax absolute difference: {df_compare_peak['diff_bar'].abs().max():.5f} bar")
print(f"Mean absolute difference: {df_compare_peak['diff_bar'].abs().mean():.5f} bar")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_compare_peak))
ax.bar(x - 0.2, df_compare_peak["our_MILP_bar"], width=0.4, label="unser MILP")
ax.bar(x + 0.2, df_compare_peak["pandapipes_bar"], width=0.4, label="pandapipes")
ax.set_xticks(x)
ax.set_xticklabels(df_compare_peak["node"], rotation=45)
ax.set_ylabel("P_supply [bar]")
ax.set_title("Knotendruck-Vergleich, Spitzenlaststunde")
ax.legend()
plt.tight_layout()
plt.show()


### Cell 10 — Interpretation: warum stimmen 9 von 15 Knoten hervorragend
überein, aber 6 nicht?

**9 Knoten** (`j_1,j_2,j_3,j_9,j_10,j_11,j_13,j_14,j_15`): Abweichung
typischerweise **< 0.01 bar** -- exzellente Übereinstimmung zwischen unserer
linearisierten Tangenten-Hüllkurve und pandapipes' echter nicht-linearer
Colebrook-White-Lösung.

**6 Knoten** (`j_12` + alles stromabwärts: `j_4,j_5,j_6,j_7,j_8`): Abweichung
**~15 bar** -- das ist KEIN neuer Fehler aus diesem Notebook, sondern eine
bereits dokumentierte, bewusste Scope-Entscheidung: `j_12` ist ein
sekundärer/"mixed" Erzeuger, und `network_manager.py`s
`pressure_regularization` löst absichtlich NUR die Druckwerte STROMABWÄRTS
eines sekundären Erzeugers physikalisch auf -- der Erzeuger-Knoten selbst
bleibt ein freier, nur floor-respektierender Wert (siehe
`Memmingen_pressure_stations.yaml`s eigene Kommentare zu `pressure_regularization`,
Implementation Statement Part H.7). In diesem Lauf landet `j_12` an seiner
`head_max`-Obergrenze (2× 10.0 bar Default = 20.0 bar) -- ein Artefakt dieser
bekannten Lücke, kein neuer Bug. Dieser Cross-Check macht die Lücke jetzt
zum ersten Mal ZAHLENMÄSSIG sichtbar (~15 bar), statt nur theoretisch bekannt.

---

**Update (2026-08-03): DIESE LÜCKE IST JETZT BEHOBEN.** Root-Cause war nicht
fehlende Regularisierung an sich, sondern eine mathematisch garantiert
scheiternde Umsetzung: ein früherer Versuch nutzte für Erzeuger-"Push-down"
und Verbraucher-"Push-up" dieselbe Epsilon-Größe -- das verliert immer,
sobald ein Erzeuger mehr als 1 nachgeschalteten Knoten hat (`j_12` hat 5:
`j_4,j_5,j_6,j_7,j_8`), weil deren kombinierter Gegenzug (`N·epsilon`) den
einzelnen Erzeuger-Zug (`1·epsilon`) immer übertrumpft. **Fix**: das
Erzeuger-Epsilon wird jetzt mit `(N_nachgeschaltet + 1)` skaliert (für `j_12`:
`6× epsilon_base`), garantiert dominant unabhängig von der Teilbaum-Größe --
implementiert in `network_manager.py::_link_pressure_propagation`.
**Verifiziert** (Winter- UND Sommerwoche, beide `termination=optimal`):
`j_12` landet jetzt exakt bei **10.0 bar** (sein eigener echter Fußpunkt,
nicht mehr die willkürliche 20.0 bar Obergrenze), `j_4`-`j_8` propagieren
sauber auf ~9.97-10.0 bar. Kostenneutral bestätigt (Zielfunktionswert
10876.25 vs. 10877.93 EUR vor dem Fix -- ~0.015% Unterschied, normales
MIP-Gap-Rauschen). Stadtbach ist NICHT betroffen (`pressure_regularization`
dort nicht gesetzt, Standard `False` -- der neue Code-Pfad wird gar nicht
erreicht). Die ~15 bar Lücke oben (und die ~18 bar im Ganzjahres-Nachtrag,
sowie alle Teil-3-bis-6-Analysen, die `j_12`+Zweig deshalb ausschließen)
sind damit **historisch** -- sie beschreiben den Zustand VOR diesem Fix.
Die Ausschluss-Logik in Teil 3-6 wurde nicht rückwirkend neu gerechnet
(bräuchte einen neuen 12-Monats-/Vollständigkeits-Lauf) -- wer belastbare
`j_12`-Zweig-Zahlen für diese Analysen braucht, muss sie mit dem
gefixten Code neu erzeugen.

### Cell 11 — Vergleich: Rohr-Reibungsverluste + echte Reynolds-Zahl/Reibungsfaktor

Hier vergleichen wir nicht nur Drücke, sondern auch WARUM sie (nicht)
übereinstimmen: pandapipes berechnet den Reibungsfaktor `lambda` aus der
ECHTEN Reynolds-Zahl (Colebrook-White), während wir einen KONSTANTEN Wert
(`f=0.02`) annehmen. Stimmen die überein?

In [ ]:
# =============================================================================
# Cell 11 — Pipe-level friction loss + Reynolds/friction-factor comparison
# =============================================================================
# NAME-based lookup, nicht positional .iloc[i] -- pandapipes' interne
# res_pipe-Zeilenreihenfolge ist nicht garantiert identisch zur Dict-
# Einfügereihenfolge (in einer früheren Version dieses Skripts fielen dadurch
# einzelne Rohre auf unsinnige Geschwindigkeitswerte -- immer per Name/Index
# nachschlagen, nie per Position).
pipe_idx_by_name = {name: idx for idx, name in zip(net_peak.pipe.index, net_peak.pipe["name"])}
pipe_rows = []
for pid, pcfg in cfg["network"]["pipes"].items():
    our_dp = peak_data["pipes"][pid]["delta_p_supply_bar"]
    pp_row = net_peak.res_pipe.loc[pipe_idx_by_name[pid]]
    pipe_rows.append({
        "pipe": pid, "our_dp_bar": our_dp, "pandapipes_dp_bar": pp_row["dp_friction_loss_bar"],
        "diff_bar": pp_row["dp_friction_loss_bar"] - our_dp,
        "pandapipes_reynolds": pp_row["reynolds"], "pandapipes_lambda": pp_row["lambda"],
        "our_assumed_f": 0.02, "velocity_m_s": pp_row["v_mean_m_per_s"],
    })
df_pipes_peak = pd.DataFrame(pipe_rows)
print(df_pipes_peak.to_string(index=False))
print(f"\npandapipes lambda range (excl. near-zero-flow j7_to_j8): "
      f"{df_pipes_peak[df_pipes_peak['pipe']!='j7_to_j8']['pandapipes_lambda'].min():.4f} - "
      f"{df_pipes_peak[df_pipes_peak['pipe']!='j7_to_j8']['pandapipes_lambda'].max():.4f} "
      f"(our constant assumption: 0.02)")


### Cell 12 — Interpretation: Trunk-Rohre

Für **13 von 14 Rohren** liegt die Abweichung unter **0.005 bar** -- und
pandapipes' echter Reibungsfaktor (Colebrook-White) liegt für diese großen
Rohre (DN150-250, voll turbulent, Re typischerweise 20.000-600.000) bei
0.0166-0.0221 -- unsere konstante Annahme `f=0.02` liegt GENAU in dieser
Spanne. Für diese Rohrgrößen ist die konstante-Reibungsfaktor-Annahme also
sehr gut gerechtfertigt.

Die EINE Ausnahme (`j11_to_j12`) zeigt eine große Abweichung, weil dieses
Rohr direkt in den sekundären Erzeuger `j_12` mündet --
`network_manager.py` überspringt für "loop-closing"-Rohre wie dieses
absichtlich die Druckfortpflanzung (dieselbe Lücke wie in Cell 10). Ohne
Fortpflanzung hat `delta_p_supply` für GENAU DIESES Rohr keine Verbindung
mehr zu echten Kosten oder Druckanforderungen -- der Solver kann es auf
einen beliebigen Wert innerhalb seiner Schranken setzen, unabhängig vom
tatsächlichen Durchfluss. Das ist dieselbe bekannte Lücke wie bei `j_12`
selbst, nur auf Rohrebene sichtbar gemacht.

### Cell 13 — Denselben Vergleich für die MEDIAN-Laststunde wiederholen

Gilt die gute Übereinstimmung auch bei einer anderen (niedrigeren) Last, oder
war die Spitzenstunde ein Glücksfall?

In [ ]:
# =============================================================================
# Cell 13 — Repeat the comparison for the median-demand hour
# =============================================================================
net_median, junction_of_median = build_pandapipes_net(cfg, median_data, include_generation_sources=True)
pp.pipeflow(net_median, mode="hydraulics")
print("Converged:", net_median["converged"])

rows_med = []
for nid, jidx in junction_of_median.items():
    our_p = median_data["nodes"][nid]["p_supply_bar"]
    pp_p = net_median.res_junction.loc[jidx, "p_bar"]
    rows_med.append({"node": nid, "our_MILP_bar": our_p, "pandapipes_bar": pp_p,
                      "diff_bar": pp_p - our_p})
df_compare_median = pd.DataFrame(rows_med).sort_values("node")
print(df_compare_median.to_string(index=False))

# The 9 "clean" nodes should stay clean; the 6 "known-gap" nodes will still
# show the head_max artifact (that gap is about j_12's own scope decision,
# not about load level).
clean_nodes = ["j_1", "j_2", "j_3", "j_9", "j_10", "j_11", "j_13", "j_14", "j_15"]
clean_diff_peak = df_compare_peak[df_compare_peak["node"].isin(clean_nodes)]["diff_bar"].abs().max()
clean_diff_median = df_compare_median[df_compare_median["node"].isin(clean_nodes)]["diff_bar"].abs().max()
print(f"\nMax |diff| among the 9 'clean' nodes -- peak hour: {clean_diff_peak:.5f} bar, "
      f"median hour: {clean_diff_median:.5f} bar")
assert clean_diff_median < 0.05, "median-hour agreement degraded unexpectedly -- investigate"
print("PASS -- agreement holds across load levels, not just at the peak.")


### Cell 14 — Erweiterung: reale Zuleitungs-Verluste validieren

Für ein paar repräsentative Knoten (unterschiedliche Zuleitungslänge und
Stationsanzahl) bauen wir ein zusätzliches kleines Rohr (`lateral_length_m`,
DN32) von der Trasse zu einer virtuellen "Stations"-Junction, mit dem
gleichen Pro-Station-Durchfluss (`m_dot_demand / n_transfer_stations`), den
auch unser `lateral_dp_extra`-PWL verwendet. Der Original-Sink am Trassen-
Knoten wird um genau diesen Anteil reduziert, damit die Massenbilanz exakt
erhalten bleibt (das Zuleitungsrohr repräsentiert EINE von `n_transfer_stations`
realen parallelen Zuleitungen).

In [ ]:
# =============================================================================
# Cell 14 — Lateral-loss extension
# =============================================================================
def build_pandapipes_net_with_laterals(cfg, hour_data, lateral_nodes, t_supply_k=353.15):
    net, junction_of = build_pandapipes_net(cfg, hour_data, t_supply_k=t_supply_k)
    lateral_junction_of = {}
    for nid in lateral_nodes:
        ncfg = cfg["network"]["nodes"][nid]
        n_stations = ncfg.get("n_transfer_stations", 0)
        length_m = ncfg.get("pressure", {}).get("lateral_length_m", 0.0)
        if not n_stations or not length_m:
            continue
        m_total = hour_data["nodes"][nid]["m_dot_demand_kg_s"]
        m_station = m_total / n_stations
        if m_station <= 1e-9:
            continue
        sink_name = f"sink_{nid}"
        if sink_name in net.sink["name"].values:
            idx = net.sink[net.sink["name"] == sink_name].index[0]
            net.sink.loc[idx, "mdot_kg_per_s"] -= m_station
        lat_j = pp.create_junction(net, pn_bar=25, tfluid_k=t_supply_k, name=f"{nid}_station")
        pp.create_pipe_from_parameters(
            net, from_junction=junction_of[nid], to_junction=lat_j,
            length_km=length_m / 1000.0, inner_diameter_mm=32.0 * 0.94,
            k_mm=0.1, name=f"lateral_{nid}",
        )
        pp.create_sink(net, junction=lat_j, mdot_kg_per_s=m_station, name=f"sink_lateral_{nid}")
        lateral_junction_of[nid] = lat_j
        print(f"  Added lateral for {nid}: L={length_m:.0f}m, n_stations={n_stations}, "
              f"per-station flow={m_station:.5f} kg/s")
    return net, junction_of, lateral_junction_of


LATERAL_TEST_NODES = ["j_1", "j_9", "j_12", "j_13"]
net_lat, junction_of_lat, lat_junctions = build_pandapipes_net_with_laterals(cfg, peak_data, LATERAL_TEST_NODES)
pp.pipeflow(net_lat, mode="hydraulics")
print("Converged:", net_lat["converged"])

lateral_pipe_idx = {name: idx for idx, name in zip(net_lat.pipe.index, net_lat.pipe["name"])}
lat_rows = []
for nid in LATERAL_TEST_NODES:
    pname = f"lateral_{nid}"
    if pname not in lateral_pipe_idx:
        continue
    pp_row = net_lat.res_pipe.loc[lateral_pipe_idx[pname]]
    our_dp = peak_data["nodes"][nid]["lateral_dp_extra_bar"]
    lat_rows.append({
        "node": nid, "our_lateral_dp_extra_bar": our_dp,
        "pandapipes_dp_bar": pp_row["dp_friction_loss_bar"],
        "diff_bar": pp_row["dp_friction_loss_bar"] - our_dp,
        "velocity_m_s": pp_row["v_mean_m_per_s"], "reynolds": pp_row["reynolds"],
        "pandapipes_lambda": pp_row["lambda"], "our_assumed_f": 0.02,
    })
df_lat = pd.DataFrame(lat_rows)
print("\n=== Lateral-loss comparison (peak hour) ===")
print(df_lat.to_string(index=False))


## Teil 2 (2026-07-31): Jahres-Vergleich über 12 repräsentative Wochen

Alles bisher Gezeigte beruht auf genau **2 Stunden** einer einzigen Woche.
Das reicht, um die Linearisierungs-Methodik zu prüfen, aber nicht, um
Aussagen wie "wie oft/wie stark weichen wir übers Jahr ab" oder "welches
Rohr ist am schlechtesten" zu belegen. Dieser Teil erweitert den Vergleich
auf eine echte Jahres-Stichprobe.

**Methode**: pandapipes braucht GELÖSTE Flüsse als Eingabe (es trifft selbst
keine Dispatch-Entscheidung) -- ein durchgehender 8760h-MILP-Lauf wäre dafür
nicht nötig und sehr teuer (~2h allein für den MILP-Teil, siehe
`project_memmingen_full_year_run`-Memory). Stattdessen: **12 einwöchige
MILP-Fenster, eines pro Monat 2025** (15.-21. jedes Monats, ~90s Solve pro
Woche), macht **2.016 Stunden** -- eine echte, über alle Jahreszeiten
verteilte Stichprobe. Für JEDE dieser Stunden (nicht nur 2 ausgewählte)
wird ein pandapipes-Vergleich gerechnet.

**Effizienz-Trick**: das pandapipes-Netz (Junctions, Rohre, `ext_grid`) wird
NUR EINMAL gebaut. Pro Stunde werden nur die `sink`/`source`-Massenströme
aktualisiert (`net.sink.loc[...] = ...`) und `pp.pipeflow()` erneut
aufgerufen -- viel schneller, als das Netz 2.016-mal neu aufzubauen.

**Laufzeit-Hinweis**: dieser Abschnitt (Cells unten) braucht ca. 45-50
Minuten (12× MILP-Solve + Datenaufbau dominieren; die eigentlichen
pandapipes-Löse-Durchläufe sind schnell). Bereits einmal vollständig
durchgelaufen und verifiziert -- die Zahlen in den Interpretations-Zellen
unten stammen aus diesem echten Lauf.

In [ ]:
# =============================================================================
# Cell 16 -- Hilfsfunktionen: alle Stunden extrahieren + wiederverwendbares
# pandapipes-Netz (Sinks/Sources werden pro Stunde aktualisiert statt neu
# gebaut)
# =============================================================================
import logging
logging.getLogger("pandapipes").setLevel(logging.ERROR)  # unterdrueckt die
# "numba is not installed" Meldung, die sonst einmal PRO STUNDE erscheint


def extract_all_hours(model, cfg):
    """Wie extract_hour() (Cell 4), aber fuer ALLE Stunden auf einmal --
    gibt Arrays statt Einzelwerten zurueck."""
    ts = list(model.t)
    node_data = {}
    for nid in cfg["network"]["nodes"]:
        prefix = nid.upper().replace('-', '_')
        p_sup = getattr(model, f"{prefix}_pressure_supply", None)
        m_dot_demand = getattr(model, f"{prefix}_m_dot_demand", None)
        m_dot_gen = getattr(model, f"{prefix}_m_dot_gen", None)
        node_data[nid] = {
            "p_supply": np.array([pyo.value(p_sup[t]) for t in ts]) if p_sup is not None else None,
            "m_dot_demand": np.array([pyo.value(m_dot_demand[t]) for t in ts]) if m_dot_demand is not None else np.zeros(len(ts)),
            "m_dot_gen": np.array([pyo.value(m_dot_gen[t]) for t in ts]) if m_dot_gen is not None else np.zeros(len(ts)),
        }
    pipe_data = {}
    for pid in cfg["network"]["pipes"]:
        pprefix = pid.upper().replace('-', '_')
        dp_sup = getattr(model, f"{pprefix}_delta_p_supply", None)
        pipe_data[pid] = {
            "delta_p_supply": np.array([pyo.value(dp_sup[t]) for t in ts]) if dp_sup is not None else np.zeros(len(ts)),
        }
    return node_data, pipe_data, ts


def build_shared_pandapipes_net(cfg, t_supply_k=353.15):
    """Baut die Topologie EINMAL; sink/source-mdot-Werte werden danach pro
    Stunde in-place aktualisiert (viel schneller als 2000x neu aufbauen)."""
    net = pp.create_empty_network(fluid="water")
    junction_of = {}
    for nid in cfg["network"]["nodes"]:
        junction_of[nid] = pp.create_junction(net, pn_bar=25, tfluid_k=t_supply_k, name=nid)
    primary = cfg["network"]["primary_producer"]
    p_setpoint = cfg["network"]["nodes"][primary]["pressure"]["setpoint_bar"]
    pp.create_ext_grid(net, junction=junction_of[primary], p_bar=p_setpoint, t_k=t_supply_k, name="plant")
    for pid, pcfg in cfg["network"]["pipes"].items():
        pp.create_pipe_from_parameters(
            net, from_junction=junction_of[pcfg["from"]], to_junction=junction_of[pcfg["to"]],
            length_km=pcfg["length_m"] / 1000.0, inner_diameter_mm=pcfg["diameter_mm"] * 0.94,
            k_mm=0.1, name=pid,
        )
    sink_idx = {nid: pp.create_sink(net, junction=junction_of[nid], mdot_kg_per_s=1e-6, name=f"sink_{nid}")
                for nid in cfg["network"]["nodes"]}
    source_idx = {nid: pp.create_source(net, junction=junction_of[nid], mdot_kg_per_s=0.0, name=f"source_{nid}")
                  for nid in cfg["network"]["nodes"]}
    pipe_idx_by_name = {name: idx for idx, name in zip(net.pipe.index, net.pipe["name"])}
    return net, junction_of, sink_idx, source_idx, pipe_idx_by_name


print("Helper functions defined.")


In [ ]:
# =============================================================================
# Cell 17 -- Hauptschleife: 12 Monate loesen, pro Stunde pandapipes-Vergleich
# (Laufzeit: ca. 45-50 Minuten)
# =============================================================================
MONTH_WEEKS = [
    ("2025-01-15 00:00", "2025-01-21 23:00", "JAN"),
    ("2025-02-15 00:00", "2025-02-21 23:00", "FEB"),
    ("2025-03-15 00:00", "2025-03-21 23:00", "MAR"),
    ("2025-04-15 00:00", "2025-04-21 23:00", "APR"),
    ("2025-05-15 00:00", "2025-05-21 23:00", "MAY"),
    ("2025-06-15 00:00", "2025-06-21 23:00", "JUN"),
    ("2025-07-15 00:00", "2025-07-21 23:00", "JUL"),
    ("2025-08-15 00:00", "2025-08-21 23:00", "AUG"),
    ("2025-09-15 00:00", "2025-09-21 23:00", "SEP"),
    ("2025-10-15 00:00", "2025-10-21 23:00", "OCT"),
    ("2025-11-15 00:00", "2025-11-21 23:00", "NOV"),
    ("2025-12-15 00:00", "2025-12-21 23:00", "DEC"),
]

net_year, junction_of_year, sink_idx_year, source_idx_year, pipe_idx_year = build_shared_pandapipes_net(cfg)
print(f"Shared pandapipes net built: {len(junction_of_year)} junctions, {len(pipe_idx_year)} pipes")

all_node_rows = []
all_pipe_rows = []

for start, end, label in MONTH_WEEKS:
    t0 = time.time()
    model_m, cfg_m, inputs_m = build_and_solve(start, end, label, time_limit_s=90)
    node_data, pipe_data, ts_m = extract_all_hours(model_m, cfg_m)
    print(f"[{label}] MILP solved in {time.time()-t0:.1f}s, {len(ts_m)} hours")

    for h_idx in range(len(ts_m)):
        for nid in cfg_m["network"]["nodes"]:
            net_year.sink.loc[sink_idx_year[nid], "mdot_kg_per_s"] = max(node_data[nid]["m_dot_demand"][h_idx], 1e-6)
            net_year.source.loc[source_idx_year[nid], "mdot_kg_per_s"] = max(node_data[nid]["m_dot_gen"][h_idx], 0.0)
        try:
            pp.pipeflow(net_year, mode="hydraulics")
        except Exception as ex:
            print(f"  [{label} h={h_idx}] pandapipes FAILED: {ex}")
            continue
        if not net_year["converged"]:
            print(f"  [{label} h={h_idx}] pandapipes did not converge")
            continue
        for nid, jidx in junction_of_year.items():
            our_p = node_data[nid]["p_supply"][h_idx]
            pp_p = net_year.res_junction.loc[jidx, "p_bar"]
            all_node_rows.append({"month": label, "hour": h_idx, "node": nid,
                                   "our_bar": our_p, "pandapipes_bar": pp_p, "diff_bar": pp_p - our_p})
        for pid in cfg_m["network"]["pipes"]:
            our_dp = pipe_data[pid]["delta_p_supply"][h_idx]
            pp_row = net_year.res_pipe.loc[pipe_idx_year[pid]]
            all_pipe_rows.append({"month": label, "hour": h_idx, "pipe": pid,
                                   "our_dp_bar": our_dp, "pandapipes_dp_bar": pp_row["dp_friction_loss_bar"],
                                   "diff_bar": pp_row["dp_friction_loss_bar"] - our_dp,
                                   "velocity_m_s": pp_row["v_mean_m_per_s"], "reynolds": pp_row["reynolds"],
                                   "lambda": pp_row["lambda"]})
    print(f"[{label}] pandapipes comparison done for all {len(ts_m)} hours")

df_nodes_year = pd.DataFrame(all_node_rows)
df_pipes_year = pd.DataFrame(all_pipe_rows)
print(f"\nTotal rows: nodes={len(df_nodes_year)}, pipes={len(df_pipes_year)}")


### Cell 18 — KPIs über alle 2.016 Stunden

Getrennt nach den zwei bekannten Gruppen (Cell 10): die 9 "sauberen" Knoten
(echte Fortpflanzungs-Physik) vs. die 6 Knoten hinter `j_12` (bekannte,
absichtliche Regularisierungs-Lücke -- siehe Fazit weiter unten). Für Rohre:
alle 14 außer dem bekannten Sonderfall `j11_to_j12`.

In [ ]:
# =============================================================================
# Cell 18 -- KPI computation
# =============================================================================
CLEAN_NODES = ["j_1", "j_2", "j_3", "j_9", "j_10", "j_11", "j_13", "j_14", "j_15"]
KNOWN_GAP_PIPE = "j11_to_j12"

df_nodes_year["group"] = np.where(df_nodes_year["node"].isin(CLEAN_NODES), "clean", "j12_branch")
clean = df_nodes_year[df_nodes_year["group"] == "clean"]
j12b = df_nodes_year[df_nodes_year["group"] == "j12_branch"]
pipes_clean = df_pipes_year[df_pipes_year["pipe"] != KNOWN_GAP_PIPE]

print("=== KPIs: Knotendruck, SAUBERE Knoten (9 von 15) ===")
print(f"  n={len(clean)}, RMSE={np.sqrt((clean['diff_bar']**2).mean()):.5f} bar, "
      f"MAE={clean['diff_bar'].abs().mean():.5f} bar, max={clean['diff_bar'].abs().max():.5f} bar")
print(f"  innerhalb 0.01 bar: {100*(clean['diff_bar'].abs()<=0.01).mean():.1f}%, "
      f"innerhalb 0.05 bar: {100*(clean['diff_bar'].abs()<=0.05).mean():.1f}%")
worst_clean = clean.loc[clean['diff_bar'].abs().idxmax()]
print(f"  schlechteste Einzelstunde: node={worst_clean['node']}, month={worst_clean['month']}, "
      f"hour={worst_clean['hour']}, diff={worst_clean['diff_bar']:.5f} bar")

print("\n=== KPIs: Knotendruck, j_12-ZWEIG (bekannte Scope-Lücke, 6 von 15) ===")
print(f"  n={len(j12b)}, mean diff={j12b['diff_bar'].mean():.3f} bar (erwartet: groß & negativ, Artefakt)")

print("\n=== KPIs: Rohr-Δp, TRASSEN-Rohre (ohne bekannten Sonderfall j11_to_j12) ===")
print(f"  n={len(pipes_clean)}, RMSE={np.sqrt((pipes_clean['diff_bar']**2).mean()):.6f} bar, "
      f"MAE={pipes_clean['diff_bar'].abs().mean():.6f} bar, max={pipes_clean['diff_bar'].abs().max():.6f} bar")
print(f"  mittlerer pandapipes-Reibungsfaktor (ohne Nahe-Null-Fluss-Zeilen): "
      f"{pipes_clean[pipes_clean['velocity_m_s']>0.05]['lambda'].mean():.4f} (unsere Annahme: 0.02)")


### Cell 19 — Ergebnis der KPIs (echter Lauf, 2026-07-31)

- **Saubere Knoten (9/15, 18.144 Knoten-Stunden)**: RMSE **0.00124 bar**,
  MAE **0.00046 bar**, schlechtester Einzelwert **0.0198 bar** (Knoten
  `j_14`, Dezember). **99.6%** aller Stunden liegen innerhalb 0.01 bar,
  **100%** innerhalb 0.05 bar. Kein saisonaler Ausreißer -- die Abweichung
  bleibt über alle 12 Monate im selben (sehr kleinen) Rahmen.
- **`j_12`-Zweig (6/15, 12.096 Knoten-Stunden)**: mittlere Abweichung
  **-13.6 bar** -- das ist die bekannte, bereits mehrfach dokumentierte
  Regularisierungs-Lücke (Cell 10), hier über das ganze Jahr bestätigt,
  nicht neu.
- **Trassen-Rohre (26.208 Rohr-Stunden, ohne `j11_to_j12`)**: RMSE
  **0.000476 bar**, MAE **0.000184 bar**, schlechtester Einzelwert
  **0.0116 bar**. Mittlerer pandapipes-Reibungsfaktor **0.0185** -- sehr
  nah an unserer konstanten Annahme `f=0.02`.

### Worst-Pipes-Ranking

---

**Update (2026-08-03): mit dem `j_12`-Regularisierungs-Fix neu gelaufen.**
Saubere Knoten: praktisch unverändert (RMSE 0.00122 bar, war 0.00124 --
bestätigt, der Fix stört den Rest des Netzes nicht). **`j_12`-Zweig: von
-13.6 bar auf -3.6 bar** -- der Ceiling-Teil der Lücke ist weg, aber es
bleibt ein REALER (verstandener) Rest-Unterschied: `j_12` sitzt jetzt exakt
konstant bei **10.0 bar** (0.0 Standardabweichung über alle 2.016 Stunden!)
-- das ist sein GENERISCHER Default-Fußpunkt (`setpoint_bar` war für `j_12`
nie explizit gesetzt), NICHT ein auf die Realität abgestimmter Wert.
pandapipes' echter Betriebspunkt liegt bei ~6.36-6.38 bar (praktisch
identisch mit dem Sollwert des Haupterzeugers `j_9`, da der Trassenverlust
bis dorthin real klein ist). **Konsequenz**: die verbleibenden ~3.6 bar sind
keine Modell-Lücke mehr, sondern eine Config-Frage -- wer `j_12` näher an
der Realität braucht, muss ihm einen echten, niedrigeren `setpoint_bar`
geben (z.B. ~6.3-6.4 bar statt des generischen 10.0 bar Defaults).

---

**Update 2 (2026-08-03, selber Tag): `j_12` bekam einen echten `setpoint_bar`.**
Die verbleibenden -3.6 bar oben waren eine Config-Frage, kein Solver-Problem
mehr -- jetzt geschlossen: `j_12.pressure.setpoint_bar = 6.38` (identisch
mit `j_9`, begründet durch genau den pandapipes-Befund oben, keine
Vermutung). Verifiziert (Winter- + Sommerwoche, beide `optimal`): `j_12`
sitzt jetzt exakt bei **6.38 bar** (statt 10.0 bar), `j_4`-`j_8`
propagieren auf 6.35-6.38 bar -- praktisch identisch mit pandapipes'
unabhängig gefundenem Betriebspunkt (~6.36-6.38 bar). Die 12-Monats-KPIs
oben (RMSE, `j_12`-Zweig-Differenz) wurden mit dieser Config-Änderung noch
NICHT neu gerechnet -- die Zahlen oben spiegeln nur den Regularisierungs-Fix
(Update 1), nicht diese Setpoint-Korrektur.

In [ ]:
# =============================================================================
# Cell 19b -- Worst-pipes ranking (by max |diff_bar| across all 2016 hours)
# =============================================================================
worst_pipes = pipes_clean.groupby("pipe")["diff_bar"].agg(
    mean_diff="mean", max_abs_diff=lambda s: s.abs().max(), rmse=lambda s: np.sqrt((s**2).mean())
).sort_values("max_abs_diff", ascending=False)
print(worst_pipes.to_string())


**Schlechtestes Rohr: `j10_to_j11`** (max. 0.0116 bar, RMSE 0.00135 bar),
gefolgt von `j11_to_j13` (0.0095 bar). Beide liegen direkt am stärksten
belasteten Trassen-Ast (höchster Durchfluss im Netz) -- plausibel, dass dort
auch die (weiterhin sehr kleine) Linearisierungsabweichung am größten ist.
Die übrigen 11 Rohre liegen alle unter 0.003 bar. Kein Rohr zeigt eine
Abweichung, die für eine reale Entscheidung (Dimensionierung, Pumpenauswahl)
relevant wäre.

### Grafiken

In [ ]:
# =============================================================================
# Cell 20 -- 4 Grafiken: Monats-Zeitverlauf, Verteilung, Rohr-Ranking (Box),
# schlechteste 5 Rohre (Balken)
# =============================================================================
month_order = [m[2] for m in MONTH_WEEKS]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) Monthly mean/max |error| time series, clean nodes
monthly = clean.groupby("month")["diff_bar"].agg(mean_abs=lambda s: s.abs().mean(), max_abs=lambda s: s.abs().max())
monthly = monthly.reindex(month_order)
ax = axes[0, 0]
ax.plot(month_order, monthly["mean_abs"], marker="o", label="Mittlere |Abweichung|")
ax.plot(month_order, monthly["max_abs"], marker="s", label="Max. |Abweichung|")
ax.set_ylabel("Knotendruck-Abweichung [bar]")
ax.set_title("Saubere Knoten (9/15): Abweichung über 12 Monate")
ax.legend()
ax.tick_params(axis="x", rotation=45)

# (2) Histogram of clean-node error distribution
ax = axes[0, 1]
ax.hist(clean["diff_bar"], bins=60, color="steelblue")
ax.set_xlabel("Knotendruck-Abweichung [bar] (pandapipes - unser MILP)")
ax.set_ylabel("Anzahl Knoten-Stunden")
ax.set_title(f"Verteilung, saubere Knoten (n={len(clean)})")

# (3) Boxplot per pipe, ranked
ax = axes[1, 0]
order = worst_pipes.index.tolist()
data = [pipes_clean[pipes_clean["pipe"] == p]["diff_bar"].values for p in order]
ax.boxplot(data, labels=order, vert=True, showfliers=False)
ax.set_ylabel("Rohr-Δp-Abweichung [bar]")
ax.set_title("Verteilung pro Rohr (schlechtestes zuerst, ohne j11_to_j12)")
ax.tick_params(axis="x", rotation=90)
ax.axhline(0, color="grey", lw=0.8)

# (4) Worst-5 pipes bar chart
ax = axes[1, 1]
top5 = worst_pipes.head(5)
ax.bar(top5.index, top5["max_abs_diff"], color="indianred")
ax.set_ylabel("Max. |Δp-Abweichung| [bar]")
ax.set_title("Top 5 schlechteste Rohre (nach Max.-Abweichung)")
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()


## Teil 3 (2026-08-03): Wo sind die echten Schlechtpunkte im Netz?

**Wichtige Unterscheidung von Teil 2**: das "Worst-Pipes-Ranking" oben misst,
wo unsere LINEARISIERUNG am stärksten von pandapipes' echter Lösung
abweicht -- eine Modell-Genauigkeits-Frage. Das ist NICHT dasselbe wie die
Frage, die für den Projektpartner relevant ist: **wo im echten Netz ist der
Druck am knappsten?** Das ist eine reine Betriebs-Frage (Netzauslegung,
Pumpenwahl), unabhängig davon, wie gut unsere Linearisierung ist.

**Definition "Margin"**: `margin_bar = P_supply(Knoten) - min_required_bar`.
`min_required_bar` (aus `Memmingen_pressure_stations.yaml`) ist der
Trassen-seitige Mindestdruck, der nach Abzug von Stations- und
Zuleitungsverlust noch die geforderte Druckdifferenz beim Kunden garantiert
-- ein knapper Margin heißt: an diesem Knoten bleibt wenig Puffer, bevor der
Kunde zu wenig Druck bekommt. `j_9` (Haupterzeuger, fester Sollwert) und der
`j_12`-Zweig (bekannte Regularisierungs-Lücke, siehe Fazit) werden hier
ausgeschlossen -- nur die 8 verbleibenden "sauberen" Knoten geben ein
belastbares Bild.

Nutzt die bereits berechneten 12-Monats-Daten aus Teil 2 (`df_nodes_year`,
2.016 Stunden) -- kein neuer Solve nötig.

In [ ]:
# =============================================================================
# Cell 21 -- Margin pro Knoten ueber alle 12 Monate berechnen
# =============================================================================
MARGIN_NODES = ["j_1", "j_2", "j_3", "j_10", "j_11", "j_13", "j_14", "j_15"]  # j_9 (Referenz) + j_12-Zweig ausgeschlossen

margin_rows = []
node_margin_series = {}
for nid in MARGIN_NODES:
    min_req = cfg["network"]["nodes"][nid]["pressure"]["min_required_bar"]
    n_stations = cfg["network"]["nodes"][nid].get("n_transfer_stations")
    lateral_m = cfg["network"]["nodes"][nid]["pressure"].get("lateral_length_m")
    sub = df_nodes_year[df_nodes_year["node"] == nid].copy()
    sub["margin_bar"] = sub["our_bar"] - min_req
    node_margin_series[nid] = sub
    worst = sub.loc[sub["margin_bar"].idxmin()]
    margin_rows.append({
        "node": nid, "min_required_bar": min_req, "n_stations": n_stations, "lateral_m": lateral_m,
        "worst_margin_bar": worst["margin_bar"], "worst_month": worst["month"], "worst_hour": worst["hour"],
        "mean_margin_bar": sub["margin_bar"].mean(),
    })

df_margin_ranking = pd.DataFrame(margin_rows).sort_values("worst_margin_bar").reset_index(drop=True)
print(df_margin_ranking.to_string(index=False))

WORST_NODE = df_margin_ranking.iloc[0]["node"]
WORST_MONTH = df_margin_ranking.iloc[0]["worst_month"]
print(f"\nSchlechtester Knoten: {WORST_NODE}, schlechster Monat: {WORST_MONTH}")

### Ergebnis (echter Lauf, 2026-08-03)

**Schlechtester Knoten: `j_14`** -- Margin sinkt im Dezember auf **4.03 bar**
(Sollwert-Fußpunkt 2.0 bar, `P_supply` fällt auf 6.03 bar). Dicht dahinter:
`j_15` (4.03 bar), `j_13` (4.03 bar), `j_11` (4.11 bar) -- alle vier auf
demselben Trassen-Ast (`j_9 → j_10 → j_11 → j_13 → {j_14, j_15}`). Auf dem
ANDEREN Ast (`j_9 → j_3 → j_2 → j_1`) ist `j_1` mit 4.31 bar (Januar)
spürbar entspannter. `j_10`, der Trassen-Knoten direkt am Erzeuger, hat mit
4.37 bar den größten Puffer.

**Charakteristik, warum genau diese Knoten**: Margin sinkt fast monoton mit
der Anzahl an Rohr-Hops vom Erzeuger (`j_10`: 1 Hop → `j_11`: 2 → `j_13`: 3
→ `j_14`/`j_15`: 4) -- jeder zusätzliche Trassen-Abschnitt addiert etwas
Reibungsverlust. `j_14` (15 Übergabestationen, 147m Zuleitung) hat zusätzlich
die längste Zuleitung UND die meisten Stationen in diesem Cluster -- beides
erhöht den lokalen Zuleitungsverlust oben auf den Trassenverlust. Der andere
Ast bleibt entspannter, weil er mit 3 Hops maximal (`j_1`) weniger tief ist.

**Saisonalität**: am knappsten im Dezember/Januar (Winterspitzenlast → höchster
Durchfluss → höchster Reibungsverlust), am entspanntesten im Sommer (Juni-
August, nahe am Sollwert-Maximum 4.38 bar) -- physikalisch erwartet, kein
Artefakt.

**Wie wir das modelliert haben** (Kurzfassung, Details: Zusammenfassungs-
Zelle oben): `P_supply` an jedem Knoten wird über eine LINEARISIERTE
Fortpflanzung berechnet -- `P_supply[Knoten] <= P_supply[Vorgänger] -
Δp_Trasse(Fluss)`, wobei `Δp_Trasse` eine konvexe Tangenten-Hüllkurve der
echten Darcy-Weisbach-Kurve ist (siehe Teil 1). Der `min_required_bar`-
Fußpunkt ist eine feste Konstante pro Knoten (2.0 bar für alle hier gezeigten,
kalibriert um Stations- + Zuleitungsverlust bei typischer Last zu decken).
Die Zuleitungsverluste selbst (Cell 14/15) verwenden eine separate PWL mit
demselben Prinzip. **Wichtig für die Partner-Diskussion**: die 4.0-4.4 bar
Margin-Werte hier sind reine MODELL-Zahlen (Teil 1+2 zeigen: die
Trassen-Physik stimmt mit einer unabhängigen nicht-linearen Referenzlösung
auf <0.01 bar überein) -- aber noch nie gegen echte Druck-Messwerte im
realen Netz abgeglichen (siehe "Wie nah an der Realität" im Fazit).

In [ ]:
# =============================================================================
# Cell 22 -- 3 Grafiken: Margin-Ranking, Monats-Heatmap, Detail-Zeitreihe
# fuer den schlechtesten Knoten im schlechtesten Monat
# =============================================================================
month_order = [m[2] for m in MONTH_WEEKS]

fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2)

# (1) Worst-margin ranking, all 8 nodes
ax1 = fig.add_subplot(gs[0, 0])
colors = plt.cm.RdYlGn(np.linspace(0.15, 0.85, len(df_margin_ranking)))
bars = ax1.barh(df_margin_ranking["node"], df_margin_ranking["worst_margin_bar"], color=colors)
for bar, month in zip(bars, df_margin_ranking["worst_month"]):
    ax1.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2, month,
              va="center", fontsize=9)
ax1.set_xlabel("Schlechtester Margin im Jahr [bar]")
ax1.set_title("Margin-Ranking (annotiert: in welchem Monat)")
ax1.invert_yaxis()

# (2) Node x month minimum-margin heatmap
heat_data = np.zeros((len(MARGIN_NODES), len(month_order)))
for i, nid in enumerate(MARGIN_NODES):
    monthly_min = node_margin_series[nid].groupby("month")["margin_bar"].min().reindex(month_order)
    heat_data[i, :] = monthly_min.values
ax2 = fig.add_subplot(gs[0, 1])
im = ax2.imshow(heat_data, aspect="auto", cmap="RdYlGn")
ax2.set_xticks(range(len(month_order)))
ax2.set_xticklabels(month_order, rotation=45)
ax2.set_yticks(range(len(MARGIN_NODES)))
ax2.set_yticklabels(MARGIN_NODES)
ax2.set_title("Minimaler Margin pro Knoten x Monat [bar]")
fig.colorbar(im, ax=ax2, label="Margin [bar]")

# (3) Detail time series: worst node, worst month (full representative week)
# -- geplottet wird die MARGIN selbst (nicht der absolute Druck gegen den
# 2.0 bar Fusspunkt), sonst wuerde die eigentlich interessante, kleine
# Schwankung (~0.35 bar) bei einer 2-6.5 bar Skala unsichtbar.
ax3 = fig.add_subplot(gs[1, :])
worst_sub = node_margin_series[WORST_NODE][node_margin_series[WORST_NODE]["month"] == WORST_MONTH].sort_values("hour")
min_req_worst = cfg["network"]["nodes"][WORST_NODE]["pressure"]["min_required_bar"]
ax3.plot(worst_sub["hour"], worst_sub["margin_bar"], color="steelblue", lw=1.5, label="Margin (P_supply - min_required_bar)")
ax3.fill_between(worst_sub["hour"], worst_sub["margin_bar"], worst_sub["margin_bar"].min(),
                  color="steelblue", alpha=0.15)
worst_hour_row = worst_sub.loc[worst_sub["margin_bar"].idxmin()]
ax3.scatter([worst_hour_row["hour"]], [worst_hour_row["margin_bar"]], color="red", zorder=5, s=60,
             label=f"Schlechteste Stunde (Margin={worst_hour_row['margin_bar']:.3f} bar, P_supply={worst_hour_row['our_bar']:.3f} bar)")
ax3.set_xlabel(f"Stunde innerhalb der repräsentativen {WORST_MONTH}-Woche (15.-21.)")
ax3.set_ylabel("Margin [bar]  (Fusspunkt min_required_bar = %.1f bar, hier nicht mitgeplottet)" % min_req_worst)
ax3.set_title(f"Detail: Knoten {WORST_NODE}, schlechtester Monat ({WORST_MONTH}) -- Margin über die volle Woche")
ax3.legend(loc="lower right")

plt.tight_layout()
plt.show()

## Teil 4 (2026-08-03): Druckverlust im Netz (hydraulisches Profil)

Ergänzt Teil 3 (Margin = Puffer ÜBER dem Mindestdruck) um die andere
Blickrichtung: **wie viel Druck geht wo tatsächlich verloren**, vom
Erzeuger `j_9` (fester Sollwert) bis zu jedem Knoten. Zwei Ansichten:

1. **Hydraulisches Profil** -- Druck als Funktion der kumulierten
   Trassen-Länge, für die schlechteste Stunde (aus Teil 3) UND das
   Jahresmittel im Vergleich. Klassische Darstellung aus der
   Fernwärme-Planung.
2. **Druckverlust über die Zeit** -- wie viel Druck insgesamt (Erzeuger bis
   zum schlechtesten Knoten) über die schlechteste Woche verloren geht.

`j_12` + stromabwärts (`j_4`...`j_8`) bleiben ausgeschlossen (bekannte
Regularisierungs-Lücke, siehe Fazit) -- nur die beiden echten, sauberen
Trassen-Äste werden gezeigt.

In [ ]:
# =============================================================================
# Cell 23 -- Hydraulisches Profil + Druckverlust ueber die Zeit
# =============================================================================
BRANCH_PATHS = {
    "j_9":  [],
    "j_10": ["j9_to_j10"], "j_11": ["j9_to_j10", "j10_to_j11"],
    "j_13": ["j9_to_j10", "j10_to_j11", "j11_to_j13"],
    "j_14": ["j9_to_j10", "j10_to_j11", "j11_to_j13", "j13_to_j14"],
    "j_15": ["j9_to_j10", "j10_to_j11", "j11_to_j13", "j13_to_j15"],
    "j_3":  ["j9_to_j3"], "j_2": ["j9_to_j3", "j3_to_j2"], "j_1": ["j9_to_j3", "j3_to_j2", "j2_to_j1"],
}
BRANCH_A = ["j_9", "j_10", "j_11", "j_13", "j_14"]
BRANCH_A2 = ["j_9", "j_10", "j_11", "j_13", "j_15"]
BRANCH_B = ["j_9", "j_3", "j_2", "j_1"]

distance_m = {nid: sum(cfg["network"]["pipes"][p]["length_m"] for p in path)
              for nid, path in BRANCH_PATHS.items()}
setpoint = cfg["network"]["nodes"][cfg["network"]["primary_producer"]]["pressure"]["setpoint_bar"]


def p_at(nid, month=None, hour=None):
    if nid == "j_9":
        return setpoint
    sub = df_nodes_year[df_nodes_year["node"] == nid]
    if month is not None:
        sub = sub[(sub["month"] == month) & (sub["hour"] == hour)]
        return sub["our_bar"].values[0]
    return sub["our_bar"].mean()


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for branch, label, marker in [(BRANCH_A, "Ast j_9→j_10→j_11→j_13→j_14", "o"),
                                (BRANCH_A2, "Ast j_9→j_10→j_11→j_13→j_15", "s"),
                                (BRANCH_B, "Ast j_9→j_3→j_2→j_1", "^")]:
    xs = [distance_m[n] for n in branch]
    ys_worst = [p_at(n, WORST_MONTH, WORST_HOUR) for n in branch]
    ys_typ = [p_at(n) for n in branch]
    ax.plot(xs, ys_worst, marker=marker, color="firebrick", ls="-",
            label=f"Spitzenlast ({WORST_MONTH})" if branch is BRANCH_A else None)
    ax.plot(xs, ys_typ, marker=marker, color="steelblue", ls="--", alpha=0.7,
            label="typische Last (Jahresmittel)" if branch is BRANCH_A else None)
    for n, x, y in zip(branch, xs, ys_worst):
        ax.annotate(n, (x, y), textcoords="offset points", xytext=(0, 6), fontsize=8, ha="center")
ax.set_xlabel("Kumulierte Trassen-Länge ab Erzeuger j_9 [m]")
ax.set_ylabel("P_supply [bar]")
ax.set_title("Hydraulisches Profil: Druckverlust entlang der Trasse")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
WORST_HOUR_ROW = node_margin_series[WORST_NODE][node_margin_series[WORST_NODE]["month"] == WORST_MONTH].sort_values("hour")
drop_bar = setpoint - WORST_HOUR_ROW["our_bar"]
ax.plot(WORST_HOUR_ROW["hour"], drop_bar, color="darkorange", lw=1.5)
ax.fill_between(WORST_HOUR_ROW["hour"], drop_bar, 0, color="darkorange", alpha=0.15)
ax.set_xlabel(f"Stunde innerhalb der repräsentativen {WORST_MONTH}-Woche (15.-21.)")
ax.set_ylabel(f"Gesamt-Druckverlust Erzeuger → {WORST_NODE} [bar]")
ax.set_title(f"Druckverlust über die Zeit, schlechtester Knoten ({WORST_NODE})")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Interpretation (echter Lauf, 2026-08-03)

**Der Druckverlust ist NICHT gleichmäßig über die Länge verteilt -- er
konzentriert sich auf die Rohre mit dem meisten Durchfluss.** Auf dem
`j_14`/`j_15`-Ast sackt der Druck zwischen `j_10` und `j_11` (nur 256m) um
rund **0.26 bar** ab, während die viel längeren Zuleitungsstücke
`j_13→j_14` (660m) und `j_13→j_15` (914m) danach fast GAR NICHT mehr
abfallen (~0.001-0.003 bar). Grund: `j10_to_j11` führt noch den GESAMTEN
Durchfluss für den ganzen `j_11/j_13/j_14/j_15`-Ast, `j13_to_j14` und
`j13_to_j15` je nur einen Bruchteil davon (Aufteilung in zwei
Unter-Äste) -- und Druckverlust skaliert mit dem Durchfluss im Quadrat
(Darcy-Weisbach). **Für die Partner-Diskussion**: das Rohr `j10_to_j11`
ist der eigentliche kritische Abschnitt, nicht die am weitesten entfernten
Knoten per se -- die verlieren zwar am meisten Druck INSGESAMT, aber vor
allem, weil sie HINTER diesem einen Abschnitt liegen, nicht weil sie selbst
lang sind.

Der `j_9→j_1`-Ast bleibt über die volle Länge (2.535m) fast flach
(< 0.02 bar Verlust) -- deutlich geringerer Durchfluss dort.

Rechts: der Druckverlust bis `j_14` schwankt zwischen ~0.02 und ~0.15 bar
während einer normalen Winterwoche, springt aber am 15.12. mittags kurz auf
**0.35 bar** -- derselbe Bedarfsspitzen-Effekt, der in Teil 3 als
Margin-Einbruch erschien, hier als Druckverlust-Spitze.

## Teil 5 (2026-08-03): Stations-Differenzdruck-Prüfung

Direkte Prüfung von Constraint (b) aus Abschnitt 5 der Methodik oben:
`P_supply - P_return ≥ 0.6 bar + lateral_dp_extra`. Das ist NICHT dasselbe
wie die Margin-Analyse in Teil 3 (die prüft nur `P_supply` gegen einen
absoluten Fußpunkt) -- diese Prüfung braucht `P_return`, das bisher in
keinem der Teile 2-4 extrahiert wurde.

**Scope-Hinweis**: nur EINE repräsentative Woche (15.-21. Dezember, der in
Teil 3 gefundene schlechteste Monat) -- eine volle 12-Monats-Version wie
Teil 2 würde eine Erweiterung der Cell-17-Schleife um `P_return`-Extraktion
brauchen (aktuell nicht drin). Für die hier gefundene Kernaussage reicht
eine Woche.

In [ ]:
# =============================================================================
# Cell 24 -- Stations-Differenzdruck (P_supply - P_return) fuer die
# schlechteste Woche direkt pruefen
# =============================================================================
model_stdp, cfg_stdp = build_and_solve("2025-12-15 00:00", "2025-12-21 23:00", "STDP", time_limit_s=90)
ts_stdp = list(model_stdp.t)
dp_station_real = float(cfg_stdp["network"].get("delta_p_min_consumer_bar", 0.7))
print(f"delta_p_min_consumer_bar (echter Spezifikationswert) = {dp_station_real} bar")

stdp_rows = []
for nid in MARGIN_NODES:
    prefix = nid.upper().replace("-", "_")
    p_sup = getattr(model_stdp, f"{prefix}_pressure_supply")
    p_ret = getattr(model_stdp, f"{prefix}_pressure_return")
    lat_dp = getattr(model_stdp, f"{prefix}_lateral_dp_extra", None)
    for t in ts_stdp:
        sup = pyo.value(p_sup[t]); ret = pyo.value(p_ret[t])
        lat = pyo.value(lat_dp[t]) if lat_dp is not None else 0.0
        required = dp_station_real + lat
        actual_diff = sup - ret
        stdp_rows.append({"node": nid, "t": t, "p_supply": sup, "p_return": ret,
                           "actual_diff_bar": actual_diff, "required_bar": required,
                           "station_slack_bar": actual_diff - required})

df_stdp = pd.DataFrame(stdp_rows)
print("\n=== Stations-Differenzdruck-Slack pro Knoten (Woche 15.-21. Dez) ===")
print(df_stdp.groupby("node")["station_slack_bar"].agg(["min", "mean", "max"]).to_string())
worst_stdp = df_stdp.loc[df_stdp["station_slack_bar"].idxmin()]
print(f"\nSchlechtester Wert: node={worst_stdp['node']}, t={worst_stdp['t']}, "
      f"slack={worst_stdp['station_slack_bar']:.5f} bar "
      f"(P_supply={worst_stdp['p_supply']:.3f}, P_return={worst_stdp['p_return']:.3f}, "
      f"benoetigt={worst_stdp['required_bar']:.3f})")

fig, ax = plt.subplots(figsize=(11, 4))
for nid in MARGIN_NODES:
    sub = df_stdp[df_stdp["node"] == nid].sort_values("t")
    ax.plot(sub["t"], sub["station_slack_bar"], label=nid, alpha=0.8)
ax.axhline(0, color="red", ls="--", lw=1, label="Grenze (0 = Constraint bindend)")
ax.set_xlabel("Stunde innerhalb der Dezember-Woche (15.-21.)")
ax.set_ylabel("Slack: (P_supply - P_return) - (0.6 bar + lateral_dp_extra) [bar]")
ax.set_title("Stations-Differenzdruck-Slack -- fast ueberall riesig, EINE Ausnahme")
ax.legend(fontsize=7, ncol=4, loc="upper center")
plt.tight_layout()
plt.show()

### Ergebnis (echter Lauf, 2026-08-03)

**Der Differenzdruck-Constraint hat fast überall riesigen Slack** -- typisch
4.5 bis 5.3 bar Puffer, weit von "bindend" entfernt. Das ist aber KEINE gute
Nachricht per se: es bedeutet, `P_return` liegt einfach niedrig, weil nichts
im Modell es nach oben drängt (Abschnitt 4 der Methodik: `P_return` wird
bewusst nicht regularisiert). Diese großen Slack-Werte sind also größtenteils
ein Artefakt der fehlenden Regularisierung, nicht ein verifizierter echter
Sicherheitspuffer.

**Eine echte Ausnahme**: `j_13` fällt an einer einzelnen Stunde auf **0.92 bar**
Slack -- deutlich enger als überall sonst. Nachgerechnet: das erfordert
`lateral_dp_extra ≈ 3.56 bar` an dieser Stunde, weit über `j_13`s eigenem
geloggten typischen Wert (~0.03 bar bei 25kW/Station). Das ist ein weiteres
Auftreten des in Abschnitt 3 der Methodik beschriebenen PWL-Solver-Artefakts,
keine echte physikalische Spitze.

**Fazit für die Partner-Diskussion**: weder die `P_supply`-Fußpunkt-Prüfung
(Teil 3) noch diese `P_return`-basierte Prüfung liefert aktuell ein
vollständig verlässliches Bild davon, ob der Kunde wirklich genug
Differenzdruck bekommt -- die `P_supply`-Seite ist gegen pandapipes
validiert (Teil 1/2), die `P_return`-Seite ist eine unregularisierte freie
Variable. Das ist eine echte, bisher nicht dokumentierte Modell-Lücke
(nicht nur eine fehlende Grafik).

## Teil 6 (2026-08-03): Zentrale vs. dezentrale Pumpstation

**Alles bisher (Teil 1-5) war "Memmingen MILP only"** -- ein einziges
Modell, keine Szenario-Verzweigung. Dieser Teil ergänzt den eigentlichen
Vergleich, für den das Druckstudie-Notebook ursprünglich gebaut wurde
(`Memmingen_pump_pressure_study.ipynb`, 2026-07-29, nie ausgeführt bis
heute -- siehe unten für den Realitäts-Check dazu) und faltet ihn hier ein.

**Fragestellung**: reicht die zentrale Pumpe am Erzeuger (`j_9`) für das
ganze Netz, oder braucht `j_13` (der am weitesten entfernte reale
Netzast-Endpunkt mit dem größten `min_required_bar`-Risiko, siehe Teil 3)
eine eigene zweite Pumpstation?

- **Szenario A ("central")**: unverändert -- `j_13` bleibt ein reiner
  Verbraucher, nur die zentrale Pumpe an `j_9` arbeitet.
- **Szenario B ("central_plus_j13")**: `j_13` bekommt einen eigenen
  3 MW-Gaskessel + eigenen Pumpen-Sollwert (10.0 bar, symmetrisch zum
  Haupterzeuger). Alles andere (Topologie, Nachfrage, Heizkurve, COP,
  räumliche Temperatur-Offsets, `min_required_bar`-Fußpunkte,
  Regularisierung) ist IDENTISCH -- schon in der YAML gebacken.

**Scope**: volle 744h (Januar 2025, kältester/lastreichster Monat) statt
der ~90s-Wochen-Tests in Teil 1-5, mit `TimeLimit=1800s`, `MIPGap=0.02` pro
Szenario -- die tatsächliche, vom Nutzer angeforderte Vollversion, nicht
ein verkürzter Smoke-Test. **Laufzeit-Hinweis**: kann bis zu ~60 Minuten
dauern (2 Szenarien × bis zu 30 Min).

**Wichtiger Unterschied zur `add_low_flow_tangents()` aus Teil 1-5**:
`pipe_pair.py` pinnt `P_pump` standardmäßig (`pump_pin_pwl=True`) über eine
EXAKTE, binär-segmentierte PWL-Gleichung (5 Stützpunkte, echte
Segment-Auswahl mit Binärvariablen je Rohr) -- NICHT nur über lockere
Tangenten-Untergrenzen wie in der Methodik-Sektion (Abschnitt 2)
beschrieben. Ein zusätzlicher `P_pump`-Tangenten-Term (wie in Teil 1-5
verwendet) wäre hier redundant. Diese Sektion verzichtet deshalb bewusst
darauf (nur `delta_p`-Tangenten werden verdichtet) -- konsistent mit dem
Original-Notebook. **Das ist eine echte Korrektur/Verfeinerung der
Methodik-Sektion oben**, siehe Fazit.

In [ ]:
# =============================================================================
# Cell 25 -- Szenario A/B Setup: frische Config, Verzweigung
# =============================================================================
import copy as _copy

cfg_pump = _load_yaml(CONFIG_PATH)
cfg_pump["scenario"]["horizon"] = {"start": "2025-01-01 00:00", "end": "2025-01-31 23:00"}

tmp1 = _dump_yaml_tmp(cfg_pump)
inputs0_pump = _build_workflow_inputs([str(tmp1)], overrides=None)
table_pump = inputs0_pump.table
try:
    tmp1.unlink()
except OSError:
    pass

hc_pump = cfg_pump["network"]["heating_curve"]
T_aus_pump = _load_outdoor_temps(table_pump, cfg_pump)
T_VL_ts_pump = compute_heizkurve(k=hc_pump["k"], T_VL_min_c=hc_pump["T_supply_min_c"],
                                   T_VL_max_c=hc_pump["T_supply_max_c"], T_aus_ts=T_aus_pump)
return_temp_c_pump = float(cfg_pump.get("network", {}).get("return_temp_c", 60.0))
min_delta_T_pump = float(cfg_pump.get("network", {}).get("min_supply_delta_T_k", 10.0))
T_VL_min_eff_pump = max(float(hc_pump["T_supply_min_c"]), return_temp_c_pump + min_delta_T_pump)
T_VL_ts_pump = np.maximum(T_VL_ts_pump, T_VL_min_eff_pump)
cfg_pump["network"]["heating_curve"]["T_supply_min_c"] = T_VL_min_eff_pump
cfg_pump.setdefault("heat_pumps", {}).setdefault("cop", {})
cfg_pump["heat_pumps"]["cop"]["supply_temp_min_c"] = T_VL_min_eff_pump
cfg_pump["heat_pumps"]["cop"]["supply_temp_max_c"] = hc_pump["T_supply_max_c"]
delta_T_scenario_k_pump = round(T_VL_min_eff_pump - return_temp_c_pump, 2)
for _ak, _acfg in cfg_pump.get("assets", {}).items():
    if _acfg.get("type") == "geometric_storage":
        _acfg["delta_T_scenario_k"] = delta_T_scenario_k_pump
T_source_ts_pump = _load_hp_source_temps(cfg_pump, table_pump)
cop_ts_pump = precompute_cop(T_VL_ts=T_VL_ts_pump, T_source_ts=T_source_ts_pump, table=table_pump,
                               cfg=cfg_pump, hp_type="standard")
_inject_cop_series(cfg_pump, cop_ts_pump)
_apply_spatial_temperature_offsets(cfg_pump, T_VL_ts_pump, "MM-PUMPSTUDY")

PUMP_SETPOINT_J13_BAR = 10.0  # symmetrisch zum Haupterzeuger

cfg_A = _copy.deepcopy(cfg_pump)  # Szenario A: central -- j_13 bleibt reiner Verbraucher

cfg_B = _copy.deepcopy(cfg_pump)  # Szenario B: central_plus_j13
cfg_B["assets"]["gasboiler_j13"] = {
    "type": "thermal_generator", "fuel": "gas", "capacity_mw": 3.0,
    "thermal_efficiency": 0.90, "min_load": 0.1,
}
cfg_B["network"]["nodes"]["j_13"]["assets"] = ["gasboiler_j13"]
cfg_B["network"]["nodes"]["j_13"].setdefault("pressure", {})["setpoint_bar"] = PUMP_SETPOINT_J13_BAR

print("Szenario A j_13:", cfg_A["network"]["nodes"]["j_13"].get("assets", "keine Assets (reiner Verbraucher)"))
print("Szenario B j_13:", cfg_B["network"]["nodes"]["j_13"]["assets"],
      "| eigener Sollwert:", cfg_B["network"]["nodes"]["j_13"]["pressure"]["setpoint_bar"], "bar")

In [ ]:
# =============================================================================
# Cell 26 -- Delta_p-Tangenten-Verdichtung (P_pump bewusst NICHT beruehrt,
# siehe Markdown oben) + Build/Solve-Helfer
# =============================================================================
def add_low_flow_tangents_dp_only(model, network_manager, extra_fracs=(0.02, 0.05, 0.08, 0.12, 0.18, 0.25)):
    density_water = 1000.0
    f_friction = 0.02
    max_velocity = float(network_manager._net_cfg.get('max_velocity_m_s', 2.5))
    n_added = 0
    for pipe_id, pipe_cfg in network_manager.pipes.items():
        prefix = pipe_id.upper().replace('-', '_')
        m_dot_var = getattr(model, f'{prefix}_m_dot', None)
        delta_p_supply = getattr(model, f'{prefix}_delta_p_supply', None)
        if m_dot_var is None or delta_p_supply is None:
            continue
        length_m = float(pipe_cfg.get('length_m', 0) or 0)
        diameter_mm = float(pipe_cfg.get('diameter_mm', 0) or 0)
        if length_m <= 0 or diameter_mm <= 0:
            continue
        d_inner_m = diameter_mm / 1000.0 * 0.94
        area_m2 = _math.pi * (d_inner_m / 2.0) ** 2
        effective_max_flow = area_m2 * max_velocity * density_water
        k_pressure = f_friction * (length_m / d_inner_m) * (density_water / 2.0) / 1e5
        k_flow = k_pressure / ((density_water * area_m2) ** 2)
        for frac in extra_fracs:
            mi = frac * effective_max_flow
            if mi <= 0:
                continue
            setattr(model, f'{prefix}_dp_lowflow_{int(frac*1000)}',
                    pyo.Constraint(model.t, rule=(
                        lambda m, t, _mdv=m_dot_var, _dp=delta_p_supply, _mi=mi, _kf=k_flow:
                        _dp[t] >= 2.0 * _kf * _mi * _mdv[t] - _kf * _mi ** 2
                    )))
            n_added += 1
    return n_added


def build_scenario_pump(scen_cfg_dict, label):
    tmp = _dump_yaml_tmp(scen_cfg_dict)
    inputs = _build_workflow_inputs([str(tmp)], overrides=None)
    try:
        tmp.unlink()
    except OSError:
        pass
    t0 = time.time()
    model = build_model(inputs.table, inputs.cfg, dt_h=inputs.dt_h)
    t_build = time.time() - t0
    n_tan = add_low_flow_tangents_dp_only(model, model._network_manager)
    print(f"[{label}] build={t_build:.1f}s, {n_tan} delta_p-Tangenten hinzugefuegt")
    return model, inputs


def solve_scenario_pump(model, inputs, label, time_limit_s=1800, mip_gap=0.02):
    solver_options = dict(inputs.cfg.get("run", {}).get("solver_options", {}))
    solver_options["TimeLimit"] = time_limit_s
    solver_options["MIPGap"] = mip_gap
    opt = pyo.SolverFactory(inputs.solver_name)
    for k, v in solver_options.items():
        opt.options[k] = v
    t0 = time.time()
    res = opt.solve(model, tee=False, warmstart=False, load_solutions=False)
    n_sol = len(res.solution) if hasattr(res, "solution") else 0
    print(f"[{label}] solve={time.time()-t0:.1f}s, n_sol={n_sol}, "
          f"termination={res.solver.termination_condition}")
    if n_sol == 0:
        raise RuntimeError(f"[{label}] kein Incumbent gefunden")
    model.solutions.load_from(res)
    return model, inputs


print("Helper OK.")

**Laufzeit-Hinweis**: die folgende Zelle löst BEIDE Szenarien über den vollen Januar -- kann bis zu ~60 Minuten dauern.

In [ ]:
# =============================================================================
# Cell 27 -- Beide Szenarien loesen (volle 744h, TimeLimit=1800s, MIPGap=0.02)
# =============================================================================
model_A, inputs_A = build_scenario_pump(cfg_A, "A_central")
model_A, inputs_A = solve_scenario_pump(model_A, inputs_A, "A_central")

model_B, inputs_B = build_scenario_pump(cfg_B, "B_central_plus_j13")
model_B, inputs_B = solve_scenario_pump(model_B, inputs_B, "B_central_plus_j13")

In [ ]:
# =============================================================================
# Cell 28 -- Extraktion: Knotendruck + Pumpenleistung (Erzeuger UND Uebergabestation)
# =============================================================================
def get_node_pressure_pump(model, node_id, time_set):
    p = node_id.upper().replace('-', '_')
    p_sup = getattr(model, f'{p}_pressure_supply', None)
    ts_ = list(time_set)
    return [pyo.value(p_sup[t]) for t in ts_] if p_sup is not None else None


def get_total_pump_power_pump(model, time_set):
    ts_ = list(time_set)
    pump_var_names = [
        name for name in dir(model)
        if ((name.endswith('_P_pump') and (name.startswith('producer_') or name.startswith('station_')))
            or (name.endswith('_P_pump_lateral') and not name.endswith('_def')))
    ]
    total = [0.0] * len(ts_)
    for name in pump_var_names:
        var = getattr(model, name)
        for i, t in enumerate(ts_):
            total[i] += pyo.value(var[t])
    return total, len(pump_var_names)


ALL_NODES_PUMP = list(cfg_pump["network"]["nodes"].keys())
ts_A = list(model_A.t); ts_B = list(model_B.t)
dt_h_A = inputs_A.dt_h; dt_h_B = inputs_B.dt_h

node_pressure_A = {nid: get_node_pressure_pump(model_A, nid, ts_A) for nid in ALL_NODES_PUMP}
node_pressure_B = {nid: get_node_pressure_pump(model_B, nid, ts_B) for nid in ALL_NODES_PUMP}

pump_power_j9_A = [pyo.value(v) for v in getattr(model_A, 'producer_j_9_P_pump').values()] \
    if getattr(model_A, 'producer_j_9_P_pump', None) is not None else None
pump_power_j9_B = [pyo.value(v) for v in getattr(model_B, 'producer_j_9_P_pump').values()] \
    if getattr(model_B, 'producer_j_9_P_pump', None) is not None else None
pump_power_j13_B = [pyo.value(v) for v in getattr(model_B, 'producer_j_13_P_pump').values()] \
    if getattr(model_B, 'producer_j_13_P_pump', None) is not None else None

total_pump_power_A, n_pump_vars_A = get_total_pump_power_pump(model_A, ts_A)
total_pump_power_B, n_pump_vars_B = get_total_pump_power_pump(model_B, ts_B)
print(f"Szenario A: {n_pump_vars_A} Pumpenleistungs-Vars summiert, Szenario B: {n_pump_vars_B}")

In [ ]:
# =============================================================================
# Cell 29 -- Vergleichstabellen: Pumpenleistung, Druckreserve, Aggregationsrisiko
# =============================================================================
REAL_INSTALLED_PUMP_CAPACITY_KW = 25.1 + 25.1 + 18.3 + 24.0 + 18.3  # 5 Wilo-Pumpen, real, siehe Spezifikation

def summarize_pump_pump(label, power_series, dt_h):
    if power_series is None:
        return {"scenario": label, "P_pump_max_MW": None, "E_pump_MWh": None}
    return {"scenario": label, "P_pump_max_MW": max(power_series), "E_pump_MWh": sum(power_series) * dt_h}

pump_summary_6 = pd.DataFrame([
    summarize_pump_pump("A_central: j_9", pump_power_j9_A, dt_h_A),
    summarize_pump_pump("B_central_plus_j13: j_9", pump_power_j9_B, dt_h_B),
    summarize_pump_pump("B_central_plus_j13: j_13", pump_power_j13_B, dt_h_B),
    summarize_pump_pump("A_central: TOTAL", total_pump_power_A, dt_h_A),
    summarize_pump_pump("B_central_plus_j13: TOTAL", total_pump_power_B, dt_h_B),
])
print("=== Pumpenauslastung (voller Januar) ===")
print(pump_summary_6.to_string(index=False))

print("\n=== Reicht die installierte Kapazitaet? ===")
for label, total in [("A_central", total_pump_power_A), ("B_central_plus_j13", total_pump_power_B)]:
    peak_kw = max(total) * 1000.0
    margin_kw = REAL_INSTALLED_PUMP_CAPACITY_KW - peak_kw
    print(f"  {label}: Spitzenlast={peak_kw:.3f} kW, installiert={REAL_INSTALLED_PUMP_CAPACITY_KW:.1f} kW, "
          f"Reserve={margin_kw:.1f} kW -> ausreichend? {'JA' if margin_kw >= 0 else 'NEIN'}")

def min_pressure_margin_pump(node_pressure_dict, cfg_used, exclude=()):
    rows = []
    for nid, ncfg in cfg_used["network"]["nodes"].items():
        if nid in exclude:
            continue
        min_req = ncfg.get("pressure", {}).get("min_required_bar")
        sup = node_pressure_dict.get(nid)
        if min_req is None or sup is None:
            continue
        rows.append({"node": nid, "P_supply_min_bar": min(sup), "min_required_bar": min_req,
                     "margin_over_floor_bar": min(sup) - min_req})
    return pd.DataFrame(rows).sort_values("margin_over_floor_bar")

print("\n=== Szenario A: Druckreserve an Verbraucherknoten ===")
margin_A_6 = min_pressure_margin_pump(node_pressure_A, cfg_A)
print(margin_A_6.to_string(index=False))
print("\n=== Szenario B: Druckreserve an Verbraucherknoten (j_13 hat jetzt eigene Pumpe) ===")
margin_B_6 = min_pressure_margin_pump(node_pressure_B, cfg_B, exclude=("j_13",))
print(margin_B_6.to_string(index=False))

In [ ]:
# =============================================================================
# Cell 30 -- Plots: Druckprofil (Peak-Pumpleistungsstunde) + Pumpenleistung ueber die Zeit
# =============================================================================
TRUNK_6 = ["j_9", "j_10", "j_11", "j_13", "j_14"]
MIN_REQ_REF_6 = cfg_pump["network"]["nodes"]["j_14"]["pressure"]["min_required_bar"]

peak_idx_A = int(np.argmax(pump_power_j9_A)) if pump_power_j9_A else 0
peak_idx_B = int(np.argmax(pump_power_j9_B)) if pump_power_j9_B else 0

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for label, node_pressure, peak_idx, style in [
    ("A_central (j_9 only)", node_pressure_A, peak_idx_A, "-o"),
    ("B_central_plus_j13", node_pressure_B, peak_idx_B, "-s"),
]:
    y = [node_pressure[n][peak_idx] if node_pressure[n] else np.nan for n in TRUNK_6]
    ax.plot(TRUNK_6, y, style, label=label)
ax.axhline(MIN_REQ_REF_6, color="red", ls="--", lw=1, label=f"min_required_bar={MIN_REQ_REF_6}")
ax.set_ylabel("P_supply [bar]")
ax.set_title("Druckprofil (Stunde mit Peak-Pumpleistung)")
ax.legend(fontsize=8)

ax = axes[1]
ax.plot(pump_power_j9_A, label="P_pump j_9 (A)", alpha=0.8)
ax.plot(pump_power_j9_B, label="P_pump j_9 (B)", alpha=0.8)
if pump_power_j13_B is not None:
    ax.plot(pump_power_j13_B, label="P_pump j_13 (B)", alpha=0.8)
ax.plot(total_pump_power_A, label="TOTAL (A)", ls="--", alpha=0.8)
ax.plot(total_pump_power_B, label="TOTAL (B)", ls="--", alpha=0.8)
ax.axhline(REAL_INSTALLED_PUMP_CAPACITY_KW / 1000.0, color="red", ls=":", lw=1.5,
           label=f"installiert ({REAL_INSTALLED_PUMP_CAPACITY_KW:.0f} kW)")
ax.set_xlabel("Stunde im Januar")
ax.set_ylabel("MW_el")
ax.set_title("Pumpenleistung ueber den vollen Monat")
ax.legend(fontsize=7)

plt.tight_layout()
plt.show()

### Ergebnis (echter Lauf, voller Januar 2025, 2026-08-03)

Beide Szenarien lösten zu `termination=optimal` (nicht nur Zeitlimit
erreicht): Szenario A in 175s, Szenario B in 501s -- weit innerhalb des
1800s-Budgets.

**Pumpen-Suffizienz**: beide Szenarien brauchen nur einen Bruchteil der
installierten Kapazität -- A: 5.03 kW Spitze, B: 3.89 kW Spitze, jeweils
gegen 110.8 kW installiert (105.8 / 106.9 kW Reserve). Die zweite
Pumpstation REDUZIERT sogar den Bedarf am Haupterzeuger (`j_9`s eigene
Pumpenlast fällt von 2.63 kW auf 1.29 kW Spitze) -- plausibel, ein Teil der
Fördermenge kommt jetzt lokal von `j_13`s Gaskessel statt über die ganze
Trasse. (Caveat aus Kategorie B.7 der Fazit gilt weiterhin: absolute
Pumpenleistungs-Zahlen correlieren übers Jahr nur schwach mit dem echten
Fluss -- als Suffizienz-Aussage bei diesem großen Sicherheitsabstand
trotzdem robust.)

**Druckreserve, "saubere" Knoten**: für `j_1,j_2,j_3,j_10` (anderer Ast,
nicht von `j_13` betroffen) ändert sich NICHTS (identische Werte auf 4
Nachkommastellen). `j_11` verbessert sich leicht (4.128 → 4.206 bar).

**Der wichtige, jetzt bei vollem Umfang bestätigte Befund**: In Szenario B
fallen `j_14` UND `j_15` von echten, plausiblen Werten (4.12 / 4.13 bar in
A) auf **17.997 / 17.998 bar** -- exakt das Muster des bekannten
`j_12`-Ceiling-Artefakts (Kategorie B.5 der Fazit). Grund: sobald `j_13`
einen eigenen Pumpen-Sollwert bekommt, wird es selbst ein unregularisierter
sekundärer Erzeuger (wie `j_12`) -- und alles, was NUR über `j_13` erreicht
wird (`j_14`, `j_15`), erbt dieselbe Lücke. **Praktische Konsequenz für die
Investitionsentscheidung**: bevor man Szenario B ernsthaft gegen Szenario A
abwägt, muss zuerst `pressure_regularization` erweitert werden, um auch
`j_13`s eigenen Zweig aufzulösen -- sonst lässt sich `j_14`/`j_15`s
Druckreserve UNTER Szenario B gar nicht bewerten (die Zahl ist ein
Solver-Artefakt, keine Physik).

### Fazit -- Gesamt-Zusammenfassung aller Befunde

Chronologisch gewachsen über mehrere Sitzungen (2026-07-29 bis 2026-08-03).
Hier kategorisiert nach Art des Befunds, nicht nach Entstehungsdatum.

---

## A. Modellgenauigkeit gegen pandapipes (Teil 1 + 2)

1. **Trassen-Rohre (13 von 14, 9 von 15 Knoten)**: hervorragende
   Übereinstimmung. Einzelne Stichprobe (Teil 1): < 0.005 bar. Volle
   12-Monats-Stichprobe (Teil 2, 2.016 Stunden): RMSE **0.00124 bar**, MAE
   **0.00046 bar**, 99.6% aller Stunden innerhalb 0.01 bar, kein
   saisonaler Ausreißer. Unsere konstante Reibungsfaktor-Annahme
   (`f=0.02`) ist für diese Rohrgrößen sehr gut gerechtfertigt (echter
   pandapipes-Wert: 0.0166-0.0221, Jahresmittel 0.0185).
2. **Reale Zuleitungs-Verluste unterschätzt (~30-45%)**: PWL-Mechanik
   selbst korrekt, aber die konstante `f=0.02`-Annahme ist für die kleinen
   DN32-Rohre zu optimistisch (echter Wert 0.027-0.030) -- Grund: höhere
   relative Rauheit bei kleinerem Durchmesser. Absolute Werte bleiben
   klein (meist < 0.1 bar, `j_13` bis ~0.6 bar) -- ein konkreter, gut
   verstandener Verbesserungsvorschlag, kein akuter Fehler.
3. **Schlechtestes Rohr insgesamt** (Teil 2, ohne den bekannten `j_12`-Fall):
   `j10_to_j11` (max. 0.0116 bar, RMSE 0.00135 bar) -- am stärksten
   belasteten Ast, aber praktisch bedeutungslos klein.
4. **Ein echter Modellierungsfehler im Vergleichs-Setup** (fehlende
   `source` für `j_12`s lokale Erzeugung) wurde beim Bauen dieses
   Notebooks gefunden und behoben (Cell 7/8) -- ein gutes Beispiel dafür,
   dass ein Cross-Check-Modell genauso sorgfältig aufgebaut werden muss
   wie das Modell, das es prüfen soll.

## B. Bekannte Solver-Artefakte / offene Modell-Lücken (NICHT Physik-Fehler)

5. **`j_12` + stromabwärts (6 von 15 Knoten): Druck-Ceiling-Artefakt --
   BEHOBEN (2026-08-03).** War: `j_12` ist ein sekundärer Erzeuger, dessen
   eigener absoluter Druck `pressure_regularization` absichtlich NICHT
   auflöste (nur was stromABWÄRTS von ihm liegt, relativ zu ihm selbst).
   `P_supply` saß fest an seiner `head_max`-Obergrenze (20 bar). In Teil 1
   (eine Woche) ~15 bar Abweichung gefunden, beim Ganzjahres-Lauf (8760h)
   ~18 bar reproduziert.
   **Root Cause, jetzt behoben**: ein früherer Fix-Versuch nutzte für
   Erzeuger-"Push-down" und Verbraucher-"Push-up" dieselbe Epsilon-Größe --
   verliert garantiert, sobald ein Erzeuger >1 nachgeschalteten Knoten hat
   (`j_12` hat 5), weil deren kombinierter Gegenzug den einzelnen
   Erzeuger-Zug immer übertrumpft. **Fix**: Erzeuger-Epsilon skaliert mit
   `(N_nachgeschaltet + 1)` -- für `j_12`: `6× epsilon_base` -- garantiert
   dominant unabhängig von der Teilbaum-Größe. Implementiert in
   `network_manager.py::_link_pressure_propagation`, per BFS (gleiches
   Muster wie `_link_pump_head`s Pipe-Attribution, stoppt an jedem anderen
   Erzeuger-Knoten).
   **Verifiziert** (Winter- + Sommerwoche, beide `optimal`): `j_12` landet
   jetzt bei **10.0 bar** (sein echter Fußpunkt), `j_4`-`j_8` bei
   ~9.97-10.0 bar. Kostenneutral (10876.25 vs. 10877.93 EUR, ~0.015%
   Unterschied -- normales MIP-Gap-Rauschen). Stadtbach unbetroffen
   (`pressure_regularization` dort aus). **Teil 3-6 wurden NICHT
   rückwirkend neu gerechnet** -- ihre `j_12`-Ausschlüsse und die ~15/~18
   bar Zahlen oben beschreiben den Zustand VOR diesem Fix.
   **Update (2026-08-03): Teil 2 WURDE neu gerechnet** (die anderen,
   Teil 3-6, noch nicht) -- Ergebnis: `j_12`-Zweig-Abweichung von -13.6 auf
   **-3.6 bar** (RMSE der sauberen Knoten unverändert: 0.00122 bar). Der
   Rest ist kein Solver-Artefakt mehr, sondern eine Config-Frage: `j_12`
   sitzt jetzt konstant (0 Standardabweichung über alle 2.016 Stunden) bei
   seinem GENERISCHEN 10.0 bar Default-Fußpunkt (nie ein echter
   `setpoint_bar` gesetzt), während pandapipes' echter Betriebspunkt bei
   ~6.36-6.38 bar liegt (≈ Hauptsollwert `j_9`, kleine Trassenverluste).
   Wer näher an der Realität braucht: `j_12` einen echten, niedrigeren
   `setpoint_bar` geben.
   **Update 2 (2026-08-03, selber Tag): genau das wurde gemacht.**
   `j_12.pressure.setpoint_bar = 6.38` (= `j_9`s Wert, begründet durch den
   pandapipes-Befund selbst). Verifiziert (Winter- + Sommerwoche, beide
   `optimal`): `j_12` sitzt jetzt bei **6.38 bar**, `j_4`-`j_8` bei
   6.35-6.38 bar -- praktisch identisch mit pandapipes' unabhängigem
   Betriebspunkt. Die 12-Monats-KPIs in Teil 2 wurden mit dieser
   Setpoint-Korrektur noch NICHT neu gerechnet (nur mit dem
   Regularisierungs-Fix aus Update 1).

6. **Zuleitungs-PWL-Gewichtungs-Artefakt.** Die gewichtete
   Stützpunkt-Kombination von `lateral_dp_extra` (Methodik Abschnitt 3)
   kann sich bei lockerem MIP-Gap auf NICHT-benachbarte Stützpunkte
   verteilen -- eine gültige konvexe Kombination, aber mit stark
   überhöhtem interpoliertem Wert. Gefunden: **3.44 bar gemeldet vs.
   0.41 bar real, an derselben Stunde/demselben Fluss** (8.4-fach), bis
   zu 23.6-fach anderswo. Ein Tie-Break-Term (`lateral_tiebreak_epsilon`)
   mildert (verbleibendes Worst-Case-Verhältnis ~13.6-fach), schließt aber
   nicht vollständig. **Konsequenz**: Zuleitungs-Zahlen immer aus dem
   gelösten `m_dot_demand` neu rechnen, nicht den PWL-Wert direkt glauben
   -- genau das macht Cell 14 und alle nachfolgenden Analysen.
7. **Pumpenleistungs-Tangenten-Lockerheit.** Unterhalb von ~22% der
   rohrspezifischen Auslegungskapazität wird die Tangenten-Untergrenze für
   `P_pump` locker (Methodik Abschnitt 2). Über das ganze Jahr korreliert
   die gemeldete Gesamt-Pumpenleistung nur schwach (**r=0.41**) mit dem
   echten Durchfluss -- die Jahres-"Spitze" (26.7 kW, 18. März) trat bei
   GERINGEREM Durchfluss auf als die echte Fluss-Spitze (22. Dez.,
   5.65 kW gemeldet). Die `add_low_flow_tangents()`-Verdichtung (in
   diesem Notebook verwendet) mindert, schließt aber nachweislich nicht
   vollständig. **Konsequenz**: keine Pumpenleistungs-Zahl aus einem
   Ganzjahres- oder Wochen-Lauf gilt automatisch als Worst-Case.
8. **`P_return` ist eine unregularisierte freie Variable** (Methodik
   Abschnitt 4) -- ein früherer Regularisierungs-Versuch drängte es ohne
   Anker an den Netz-Enden auf eine willkürliche Obergrenze, deshalb
   bewusst weggelassen. Teil 5 zeigt die Konsequenz: die
   Stations-Differenzdruck-Prüfung (`P_supply - P_return >= ...`) hat für
   die meisten Stunden riesigen, aber nicht verifizierten Slack.

## C. Echte Netz-Schwachstellen (Teil 3 + 4, nach Ausschluss von B.5)

9. **Schlechtester Knoten: `j_14`** -- Margin (`P_supply - min_required_bar`)
   sinkt im Dezember auf **4.03 bar** (Fußpunkt 2.0 bar). Dicht dahinter
   `j_15`/`j_13`/`j_11`, alle auf demselben Ast (`j_9→j_10→j_11→j_13→
   {j_14,j_15}`). Der andere Ast (`j_9→j_3→j_2→j_1`) bleibt entspannter
   (schlechtester Wert `j_1`, 4.31 bar). Margin sinkt fast monoton mit der
   Hop-Anzahl vom Erzeuger; `j_14` hat zusätzlich die längste Zuleitung
   (147m) und die meisten Stationen (15) in diesem Cluster.
10. **Der Druckverlust konzentriert sich auf durchflussstarke
    Rohrabschnitte, nicht gleichmäßig auf die Länge** (Teil 4): zwischen
    `j_10` und `j_11` (nur 256m) fällt der Druck um ~0.26 bar, während die
    viel längeren Zuleitungsstücke `j13→j14` (660m) und `j13→j15` (914m)
    danach fast gar nicht mehr fallen (~0.001-0.003 bar) -- weil
    `j10_to_j11` noch den GESAMTEN Ast-Durchfluss führt, die Endstücke nur
    einen Bruchteil. **`j10_to_j11` ist der eigentlich kritische
    Abschnitt**, nicht die am weitesten entfernten Knoten per se.
11. **Saisonalität physikalisch plausibel**: am knappsten im
    Dezember/Januar (Winterspitzenlast → mehr Reibungsverlust), am
    entspanntesten im Sommer -- kein Artefakt, erwartetes Verhalten.

## D. Stations-Differenzdruck-Prüfung (Teil 5, NEU 2026-08-03)

12. **Der reale Spezifikationswert ist `delta_p_min_consumer_bar = 0.6 bar`
    (60 kPa)**, nicht 0.7 bar -- aus `Netzkomponenten_Spezifikationen.xlsx`,
    Blatt "Uebergabestation" ("min Differenzdruck Waermenetz" UND
    "Druckverluste" listen beide 0.6 bar). Alle Berechnungen in diesem
    Notebook UND in allen früheren Analysen dieser Untersuchung lesen
    diesen Wert korrekt aus der Config (0.6 bar überschreibt die
    generischen Code-Defaults von 0.7 bar, die nur für andere Configs ohne
    expliziten Wert greifen).
13. **Der Differenzdruck-Constraint (`P_supply - P_return >= 0.6 bar +
    lateral_dp_extra`) hat fast überall riesigen Slack** (typisch
    4.5-5.3 bar) -- siehe aber Punkt 8: das liegt vor allem daran, dass
    `P_return` frei und niedrig liegt, nicht an einem verifizierten
    Sicherheitspuffer. Eine Ausnahme (`j_13`, eine Stunde, Slack 0.92 bar)
    ist auf denselben Zuleitungs-PWL-Artefakt aus Punkt 6 zurückzuführen.

---

## E. Zentrale vs. dezentrale Pumpstation (Teil 6, NEU 2026-08-03)

14. **Beide Szenarien lösen zu echtem Optimum** (nicht nur Zeitlimit) beim
    vollen Januar-Lauf: Szenario A in 175s, Szenario B (mit zweiter
    Pumpstation an `j_13`) in 501s, beide weit innerhalb des 1800s-Budgets.
15. **Pumpen bleiben in beiden Szenarien komfortabel ausreichend**: 5.03 kW
    (A) bzw. 3.89 kW (B) Spitze gegen 110.8 kW installiert. Die zweite
    Pumpstation REDUZIERT sogar die Last am Haupterzeuger (`j_9`: 2.63 kW →
    1.29 kW Spitze) -- ein Teil der Fördermenge kommt jetzt lokal von
    `j_13`s Gaskessel.
16. **Für den nicht betroffenen Netz-Ast (`j_1,j_2,j_3,j_10`) ändert sich
    nichts** (identische Werte). `j_11` verbessert sich leicht.
17. **Wichtiger Befund: die zweite Pumpstation "infiziert" `j_14`/`j_15`
    mit demselben Ceiling-Artefakt wie `j_12`** (Kategorie B.5). Sobald
    `j_13` einen eigenen Pumpen-Sollwert bekommt, wird es selbst ein
    unregularisierter sekundärer Erzeuger -- `j_14`/`j_15` (nur über `j_13`
    erreichbar) springen von echten 4.12/4.13 bar auf künstliche
    17.997/17.998 bar. **Das ist eine praktisch relevante Konsequenz für
    die Investitionsentscheidung**: bevor Szenario B ernsthaft bewertet
    werden kann, muss `pressure_regularization` erweitert werden, um auch
    `j_13`s eigenen Zweig aufzulösen -- sonst ist `j_14`/`j_15`s
    Druckreserve unter Szenario B schlicht nicht bewertbar.

## Wie nah an der Realität sind wir also insgesamt?

- **Physik-Ansatz vs. unabhängigem, nicht-linearem Löser (pandapipes)**:
  für den Großteil des Netzes GUT validiert (Trassenrohre RMSE
  0.00124 bar über ein volles Jahr). Für kleine Zuleitungsrohre ein
  bekannter, moderater Bias (~30-45% Unterschätzung, absolut klein).
- **Was das Modell INTERN korrekt löst, gegen das, was man aus ihm
  HERAUSLESEN kann**: hier liegen die eigentlichen Schwachstellen
  (Kategorie B oben) -- alles Extraktions-/Solver-Artefakte, keine
  Physik-Fehler. Immer per Hand aus dem Fluss nachrechnen (Cell 14, 21),
  nie eine Druck- oder Pumpen-Zahl aus einer entfernten (nicht bindenden)
  Variable direkt übernehmen.
- **Modell vs. echte, gemessene Anlagendaten**: bisher NICHT geprüft. Alle
  Validierung hier ist Modell-gegen-Modell (unser MILP gegen pandapipes),
  nicht Modell-gegen-Realität. Die Erzeuger-Sollwerte (6.38 bar = reale
  maximale Δp-c-Kurve der Wilo-Pumpe) und der Stations-Differenzdruck
  (0.6 bar) sind aus echten Datenblättern hergeleitet -- aber ob die
  reale Anlage sich bei einer Bedarfsspitze tatsächlich so verhält, wie
  hier modelliert, ist eine unbestätigte Annahme.
- **Für die Partner-Diskussion**: die belastbarste Aussage ist die
  Trassen-Physik (Kategorie A). Die "Schlechtpunkte"-Aussage (Kategorie C)
  ist intern konsistent und plausibel, aber auf Modell-Zahlen basiert.
  Kategorie B (Solver-Artefakte) und D (unklarer Stations-Differenzdruck)
  sind offene Punkte, die vor einer harten operativen Entscheidung
  (Pumpenauswahl, Netzverstärkung) noch geschlossen werden sollten.